# KuaiRand-Pure：用户画像、视频供给、曝光偏差诊断与长播放预测
基于KuaiRand-Pure数据集构建用户、视频和作者历史画像，并利用同时期的随机曝光作为参照，分析标准推荐日志中的视频曝光分配与反馈差异；最后补充标准推荐场景下的long_view预测。

# Phase 0. 环境准备

In [ ]:
# 0.1 导入本次分析所需要的包：
import numpy as np
import pandas as pd
from IPython.display import display         # 在Notebook中显示DataFrame/Styler（让表格输出更美观）
from pathlib import Path                    # 文件夹和文件路径管理
import matplotlib.pyplot as plt             # 基础绘图、画布、坐标轴、保存图片
import seaborn as sns                       # 也是绘图
from scipy.stats import wilcoxon, rankdata  # Wilcoxon 配对符号秩检验       (用于Phase5 标准推荐 vs 随机曝光)
                                            # rankdata 给数值计算秩排名     (用于Phase5 秩二列效应量)
from statsmodels.stats.multitest import multipletests   # 多重假设检验校正  (用于Phase5 BH-FDR)

from sklearn.model_selection import train_test_split    # 划分训练数据（用于Phase6 长播放预测）
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score,roc_curve
                                                        # 计算ROC-AUC（用于Phase6 评价模型能力）
                                                        # 计算精确率（预测为正的样本里，真实为正的比例）
                                                        # 计算召回率（所有真实正样本里面，被预测为正的比例）
                                                        # 计算F1分数（精确率和召回率的综合情况）
                                                        # 绘制ROC曲线
import lightgbm as lgb

# 0.2 设置绘图样式
plt.rcParams.update({
    "figure.dpi": 120,           # 屏幕显示清晰度
    "savefig.dpi": 300,          # 导出图片清晰度
    "axes.titlesize": 14,        # 子图标题
    "axes.labelsize": 12,        # 坐标轴标签
    "xtick.labelsize": 10,       # X轴刻度
    "ytick.labelsize": 10,       # Y轴刻度
    "legend.fontsize": 10,       # 图例
    "figure.titlesize": 16,      # 全图总标题
})

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS", "DejaVu Sans"]                           # 设置中文字体
plt.rcParams["axes.unicode_minus"] = False      # 负号正常显示
plt.rcParams["axes.spines.top"] = False         # 去掉上边框
plt.rcParams["axes.spines.right"] = False       # 去掉右边框

# 0.3 统一项目根目录、数据目录与输出目录
PROJECT_ROOT = Path.cwd()                     # 当前工作目录
if PROJECT_ROOT.name == "notebooks":          # 若从 notebooks 目录启动，回退到项目根目录
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUT_PHASE3 = OUTPUT_ROOT / 'phase3_user'
OUT_PHASE3.mkdir(parents=True, exist_ok=True)
OUT_PHASE4 = OUTPUT_ROOT / 'phase4_video_author'
OUT_PHASE4.mkdir(parents=True, exist_ok=True)
OUT_PHASE5 = OUTPUT_ROOT / 'phase5_standard_random'
OUT_PHASE5.mkdir(parents=True, exist_ok=True)
OUT_PHASE6 = OUTPUT_ROOT / 'phase6_precision'
OUT_PHASE6.mkdir(parents=True, exist_ok=True)
print('环境配置完成！')

In [ ]:
#  0.4 MySQL数据库连接

# 数据库账号信息从项目根目录的 .env 文件读取，

# 复现项目时：
# 1. 复制 .env.example
# 2. 重命名为 .env
# 3. 填写自己本地MySQL的账号和密码
# ============================================================

import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


# （1）读取项目根目录下的.env文件
load_dotenv()

# （2）读取数据库连接参数
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "kuairand_pure")

# （3）创建数据库连接地址
database_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    query={"charset": "utf8mb4"}
)

# （4）创建SQLAlchemy Engine
engine = create_engine(
    database_url,
    pool_pre_ping=True
)

# （5）测试数据库连接
with engine.connect() as conn:
    current_database = conn.execute(
        text("SELECT DATABASE();")
    ).scalar()
print(f"MySQL连接成功，当前数据库：{current_database}")

# Phase 1. 原始数据理解与质量检查

## 1.1 构建数据字典
查看每张表的数据结构，构建数据字典，以回答以下问题：
* 每张表有多少行、多少列
* 每个字段叫什么名字，是什么数据类型
* 哪个字段可能是主键
* 不同表之间如何关联
* 每一行数据代表什么（表粒度）

该数据集共有6份文件，分别是：
* log_standard_4_08_to_4_21_pure.csv  #前期标准推荐交互日志表（时间：0408-0421）
* log_standard_4_22_to_5_08_pure.csv  #后期标准推荐交互日志表（时间：0422-0508）
* log_random_4_22_to_5_08_pure.csv    #后期随机曝光交互日志表（时间：0422-0508）
* user_features_pure.csv              #用户特征表
* video_features_basic_pure.csv       #视频基础特征表
* video_features_statistic_pure.csv   #视频统计特征表

In [ ]:
#加载各个表格
log_standard_pre = pd.read_csv(DATA_DIR / "log_standard_4_08_to_4_21_pure.csv")
log_standard_post = pd.read_csv(DATA_DIR / "log_standard_4_22_to_5_08_pure.csv")
log_random_post = pd.read_csv(DATA_DIR / "log_random_4_22_to_5_08_pure.csv")
users = pd.read_csv(DATA_DIR / "user_features_pure.csv")
video_basic = pd.read_csv(DATA_DIR / "video_features_basic_pure.csv")
video_statistic = pd.read_csv(DATA_DIR / "video_features_statistic_pure.csv")

### 1.1.1 构建日志表字典

In [ ]:
#查看各个表的结构
log_standard_pre.info()
display(log_standard_pre.head(5))
i = log_standard_pre.duplicated().sum()
j = len(log_standard_pre)
print('前期标准推荐交互日志表中共有19列字段，全部字段均为int64类型，且无缺失值')
print(f'该表中共有{j}行数据')
print(f'该表中存在{i}行重复值')

In [ ]:
log_standard_post.info()
display(log_standard_post.head(5))
i = log_standard_post.duplicated().sum()
j = len(log_standard_post)
print('后期标准推荐交互日志表中共有19列字段，全部字段均为int64类型，且无缺失值')
print(f'该表中共有{j}行数据')
print(f'该表中存在{i}行重复值')

In [ ]:
log_random_post.info()
display(log_random_post.head(5))
i = log_random_post.duplicated().sum()
j = len(log_random_post)
print('后期随机曝光交互日志表中共有19列字段，全部字段均为int64类型，且无缺失值')
print(f'该表中共有{j}行数据')
print(f'该表中存在{i}行重复值')

* 检查发现三组日志表字段结构一致,字段定义与质量检查规则见 docs/data_dictionary.md
* 日志表可通过user_id、video_id与用户维表、视频维表进行关联
* 日志表每一行的数据代表一次用户与视频的交互行为，如某用户观看一次某视频，就会产生一条记录

### 1.1.2 构建用户特征表字典

In [ ]:
users.info()
display(users.head(5))
i = users.duplicated().sum()
j = len(users)
print('用户特征表中共有31列字段，其中13列有效字段，18列匿名字段。有效字段无缺失值')
print(f'该表中共有{j}行数据')
print(f'该表中存在{i}行重复值')

* 字段定义与质量检查规则见 docs/data_dictionary.md
* 用户特征表主键应该是user_id，并通过该主键与日志表关联
* 用户特征表每一行代表了一个用户的各类信息，包括用户id、活跃程度等

### 1.1.3 构建视频基础特征表字典

In [ ]:
video_basic.info()
display(video_basic.head(5))
i = video_basic.duplicated().sum()
j = len(video_basic)
print('视频基础特征表中共12列字段，video_duration,music_type,tag字段有缺失值')
print(f'该表中共有{j}行数据')
print(f'该表中存在{i}行重复值')

* 字段定义与质量检查规则见 docs/data_dictionary.md
* 视频基础特征表主键应该是video_id并通过该主键和日志表关联
* 视频基础特征表每行代表了一个视频的基础信息，包括视频id、创作者id等

### 1.1.4 构建视频统计特征表字典

In [ ]:
video_statistic.info()
display(video_statistic.head(5))
i = video_statistic.duplicated().sum()
j = len(video_statistic)
print('视频统计特征表中共有52列字段，以数值型统计指标为主。video_id 为视频标识，其余主要记录过去一个月的曝光、播放和互动统计。')
print(f'该表中共有{j}行数据')
print(f'该表中存在{i}行重复值')

* 字段定义与质量检查规则见 docs/data_dictionary.md
* 视频统计特征表的主键应该是video_id，并通过该主键与视频基础表或曝光日志中的视频记录关联
* 视频统计特征表每行代表了一个视频更详细的信息，包括完播率、长时间播放率等

## 1.2 数据校验
* 校验表格是否存在缺失值、空值、重复值（该步骤在1.1已完成）
* 校验特殊字段（如二值字段、时间字段）是否存在异常值
* 校验关键字段（如主键）是否唯一

需要注意的是：本章节只负责发现问题，对于问题的处理（数据清洗）则是后续用SQL进行。

原因：后续需要对清洗后的表格进行联合、多表查询，如果用python清洗的话，又要额外生成新的清洗后的表格再导入SQL。还不如让原始数据进入数据库后就一直在数据库内部进行加工。

### 1.2.1 日志表字段校验

校验连续字段是否存在异常值

In [ ]:
display(log_standard_pre.describe().style.format("{:.2f}"))
display(log_standard_post.describe().style.format("{:.2f}"))
display(log_random_post.describe().style.format("{:.2f}"))
print('video_id,hourmin,play_time_ms等连续字段的取值都在合理范围内')

校验二值字段字段是否存在异常值

In [ ]:
binary_cols = ['is_click','is_like','is_follow',"is_comment","is_forward","is_hate","long_view","is_profile_enter","is_rand"]   # 日志表中的二值字段，理论上数值非0即1
tables_logs = {'log_standard_pre':log_standard_pre,'log_standard_post':log_standard_post,'log_random_post':log_random_post}     # 三个日志表

logs_check_result=[]    # 建立一个空表格储存结果

for name,df in tables_logs.items():
    for col in binary_cols:              # 取出刚刚建好的表格中的二值字段
        missing = df[col].isna().sum()   # 检查字段有无空值（虽然前面已经发现字段都无空值）
        error_num = (df[col].notna() & ~df[col].isin([0,1])).sum()  # 二值字段中非空且取值不为0，1的值
        value_distribution = df[col].value_counts(dropna=False)     # 每个二值字段的各取值总数
        logs_check_result.append({'表格':name,'检查字段':f'{col}','缺失值':int(missing),'异常值数':int(error_num),'取值分布':value_distribution})    #将结果存入刚刚简历的空表中

display(logs_check_result)
print('\n三个日志表二值字段均无异常值')

校验特殊字段是否存在异常值

In [ ]:
#校验tab字段是否存在异常值（取值范围应该是0~14之间的整数）
logs_check_result=[]                    # 建立一个空表储存结果(此处为将这个表格重新设置为空表格)

for name,df in tables_logs.items():     # 循环取出三个日志表
    missing = df['tab'].isna().sum()    # 检查字段有无空值（虽然前面已经发现字段存在空值）
    error_num = (df['tab'].notna() & ~df['tab'].between(0,14)).sum()    # 检查字段非空且不在0~14取值范围内的值
    value_distribution = df['tab'].value_counts(dropna=False)           # 每个二值字段的各取值总数
    logs_check_result.append({'表格':name,'检查字段':'tab','缺失值':missing,'异常值数':error_num,'取值分布':value_distribution}) # 将结果存入空表中

display(logs_check_result)
print('\n三个日志表tab字段均无异常值')

日志表校验结果：
* 三组日志表的字段类型、缺失情况和基础取值范围未发现明显异常，但三个日志表均存在重复记录，后续DWD层需要统一去重
* 原始long_view字段与官方定义不一致，因此后续相关分析均使用DWD层重新计算的long_view
* date和hourmin后续统一转换为标准时间字段

### 1.2.2 用户表字段校验

校验主键是否唯一

In [ ]:
userid_check = users['user_id'].duplicated().sum()
print(f'用户表主键重复数为{userid_check}，主键唯一')

校验连续字段是否存在异常值

In [ ]:
display(users[['user_id','follow_user_num','fans_user_num','friend_user_num','register_days']].describe().style.format("{:.1f}"))
print('\n连续字段取值正常')

校验二值字段是否存在异常值

In [ ]:
binary_cols = ['is_lowactive_period','is_video_author','is_live_streamer']  # 存入用户表的二值字段
users_check_result=[]   #设置一个空表存储检查结果

for col in binary_cols: # 取出刚刚存入的二值字段
    missing = users[col].isna().sum()       # 检查空值（虽然前面已经发现字段无空值）
    error_num = (users[col].notna() & ~users[col].isin([0,1])).sum()    # 检查字段中非空且取值不为0，1的值
    value_distribution = users[col].value_counts(dropna=False)          # 每个二值字段的各取值总数
    users_check_result.append({'表格':'users','检查字段':col,'缺失值':missing,'异常值数':error_num,'取值分布':value_distribution})    #将结果存入表格中

display(users_check_result)
print('\nis_live_streamer仅出现1和-124两个值，为满足后续二值分析，在数据清晰阶段将-124映射为0')

校验分类字段是否存在异常值

In [ ]:
users_check_result=[]   # 重新将表清空
category_col = ['user_active_degree','follow_user_num_range','fans_user_num_range','friend_user_num_range','register_days_range']   # 将分类字段存入变量
for col in category_col:
    missing = users[col].isna().sum()
    # 此处不检查error_num是因为每个字段的取值规则都不一样，而且直接看取值分布也可以知道有没有error_num
    value_distribution = users[col].value_counts(dropna=False)
    users_check_result.append({'表格':'users','检查字段':col,'缺失值':missing,'取值分布':value_distribution})

display(users_check_result)
print('\nuser_active_degree字段存在6个unknown值，其余字段无异常值')

用户表校验结果：
* * is_live_streamer字段仅出现1和-124，后续处理将-124映射为0
* user_active_degree存在6个unknown值

### 1.2.3 视频基础特征表字段校验

校验主键是否唯一

In [ ]:
video_id_check = video_basic['video_id'].duplicated().sum()
print(f'表中主键列（video_id）重复值为{video_id_check}')

校验连续字段是否存在异常值

In [ ]:
display(video_basic[['video_id','author_id','visible_status','video_duration','server_width','server_height','music_id']].describe().style.format('{:.2f}'))
#检查发现连续字段大部分都处于正常范围，video_duration的count数量少于总数，说明存在缺失值
print('\nvideo_duration字段存在缺失值')

校验分类字段是否存在异常值

In [ ]:
video_basic_check_result = []   # 建立空表储存结果

category_col = ['video_type','upload_type','tag','music_type','upload_dt']  # 将分类字段存入该变量
for col in category_col:
    missing = video_basic[col].isna().sum() # 检查空值（虽然前面已经检查tag,music_type存在缺失值）
    # 此处也不检查error_num，原因同上！（1是每个字段取值都不同，2是在value_distribution也能看出有没有错误值）
    value_distribution = video_basic[col].value_counts(dropna=False)
    video_basic_check_result.append({'表格':'video_basic','检查字段':col,'缺失值':missing,'取值分布':value_distribution})
display(video_basic_check_result)
print('\nvideo_type,upload_type字段存在unknow值,tag,music_type字段存在缺失值')

视频基础特征表校验结果：
* video_type,upload_type字段存在unkonwn值
* video_duration,tag,music_type字段存在缺失值

### 1.2.4 视频统计特征表字段校验

校验主键是否唯一

In [ ]:
video_id_check = video_statistic['video_id'].duplicated().sum()
print(f'视频统计特征表主键(video_id)重复值为：{video_id_check}')

校验连续字段是否存在异常值

In [ ]:
display(video_statistic.describe().style.format('{:.2f}'))
print('视频统计特征标中字段均为连续字段\n视频统计特征表不存在异常值')

至此，已完成对各表的字段校验，结论在各小节结尾处。对每个字段的校验原则已写入数据字典中。

# Phase 2. SQL：数据模型、质量验证与清洗
结合Phase 1的校验结果，利用MYSQL对原数据进行建表、清洗，并按照数仓结构建立后续所有分析所依赖的数据基座。

处理路径：
1. ODS：CSV 原样入库
2. DWD：去重、时间标准化、异常编码处理，整合三类日志为exposure_log
3. DWS：按 用户/视频/作者 × log_source 聚合关键指标并计算行为率

总共构建表格：
1. ODS层：
* ods_log_standard_pre：前期标准推荐交互日志表；
* ods_log_standard_post：后期标准推荐交互日志表；
* ods_log_random_post：后期随机曝光交互日志表；
* ods_users：用户特征表；
* ods_video_basic：视频基础特征表；
* ods_video_statistic：视频统计特征表；（本次项目分析中并没有用到）

2. DWD层：
* dwd_log_standard_pre：前期标准推荐交互日志表（清洗后）；
* dwd_log_standard_post：后期标准推荐交互日志表（清洗后）；
* dwd_log_random_post：后期随机曝光交互日志表（清洗后）；
* dim_users：用户特征表(清洗后)；
* dim_video_basic：视频基础特征表(清洗后)；
* dim_video_statistic：视频统计特征表(清洗后)；
* exposure_log：整合三张不同时期、曝光来源日志表的总日志表

3. DWS层：
* dws_user_window_metrics：按用户 × log_source 聚合的窗口级行为指标表
* dws_video_window_metrics：按视频 × log_source 聚合的窗口级行为指标表
* dws_author_window_metrics：按作者 × log_source 聚合的窗口级行为指标表

# Phase 3. 用户画像与历史特征建设

## 3.1 构建用户历史行为画像表
使用画像快照日前的历史数据，构建一套“一行一个用户”的历史画像

本阶段构建的供后续使用的表：
* bridge_video_tag:视频—标签桥接表，得到每一个视频对应的单个/多个tag（粒度：video_id × tag）
* ads_user_interest_profile：用户视频标签偏好画像表，汇总用户Top1/Top2偏好标签、对应偏好得分及得分占比等兴趣特征
* ads_user_pre_performance_feature：用户历史行为画像表，汇总用户在standard_pre窗口内的活跃、曝光、观看和互动等历史行为指标,以及用户视频标签偏好

## 3.2 用户历史行为画像分析
根据ads_user_pre_performance_feature中的字段，对用户历史行为画像分析

In [ ]:
# （1）加载表格
user_pre_profile = pd.read_sql("SELECT * FROM ads_user_pre_performance_feature",con=engine)

# （2）分析预处理
u = user_pre_profile.copy()                        # 取一个简单名字的变量复制做后续处理，不对user_pre_profile原始表格进行修改
u_active = u[u["exposures"] > 0].copy()            # 取出有历史活跃数据的用户信息

# （3）构建长尾变量坐标转换函数（用于绘图）：对非负长尾数据进行log1p转换，并生成原始数值对应的横轴刻度，避免极端大值压缩主体分布
def log1p_axis_values(data, n_ticks=6):
    values = pd.to_numeric(data, errors="coerce").dropna()
    values = values[values >= 0]
    if values.empty:
        return values, np.array([0.0]), ["0"]
    log_values = np.log1p(values)
    max_value = values.max()
    raw_ticks = np.array([0.0]) if max_value == 0 else np.expm1(np.linspace(0, np.log1p(max_value), n_ticks))
    log_ticks = np.log1p(raw_ticks)
    tick_labels = [f"{v:.0f}" for v in raw_ticks]
    return log_values, log_ticks, tick_labels
# 输入：data：待转换的非负长尾数值数据；n_ticks：横轴刻度数量，默认6个
# 输出：log_values：log1p转换后的绘图数据；log_ticks：转换后的横轴刻度位置；tick_labels：对应的原始数值刻度标签

### 3.2.1 用户历史活跃与曝光规模
分析用户的历史活跃特征、以及曝光规模

In [ ]:
# （1）用户历史活跃与曝光规模描述性统计
print(f"全量用户数={len(u)}，有历史曝光用户数={len(u_active)}（{len(u_active)/len(u)*100:.1f}%）")

display(u_active[['days_since_last_active']].describe().style.format('{:.2f}'))                   # 只看有历史活跃数据的用户

display(u[['active_days',"exposures","unique_exposed_videos"]].describe().style.format("{:.2f}")) # 对全量用户分析

In [ ]:
## 图3-1：用户历史活跃与曝光规模
fig, axes = plt.subplots(2,2,figsize=(13, 10))

## 图3-1a 历史活跃天数
sns.histplot(data=u, x="active_days", bins=np.arange(-0.5, 14.5, 1), ax=axes[0,0], edgecolor="black")                      # 绘制直方图
axes[0,0].axvline(u["active_days"].median(), color="red", linestyle="--", label=f'中位数={u["active_days"].median():.0f}')  # 绘制中位数线
axes[0,0].set(title="历史活跃天数", xlabel="活跃天数", ylabel="用户数")                                                         # 设置标题
axes[0,0].legend()                                                                                                         # 显示图例

# 图3-1b：最后活跃日距快照日天数（只保留有历史活跃天数的用户）
sns.histplot(data=u_active, x="days_since_last_active", bins=np.arange(-0.5, 14.5, 1), ax=axes[0,1], edgecolor="black")
axes[0,1].axvline(u_active["days_since_last_active"].median(), color="red", linestyle="--", label=f'中位数={u_active["days_since_last_active"].median():.0f}')
axes[0,1].set(title="最后活跃日距快照日天数", xlabel="天数", ylabel="用户数")
axes[0,1].legend()

# 图3-1c:用户历史接收视频曝光次数
exposure_log, exposure_ticks, exposure_labels = log1p_axis_values(u["exposures"])   # 该字段呈长尾分布，进行log1p转换以减弱极端值对横轴展示的影响
sns.histplot(x=exposure_log, bins=20, ax=axes[1,0], edgecolor="black")
axes[1,0].axvline(np.log1p(u["exposures"].median()), color="red", linestyle="--", label=f'中位数={u["exposures"].median():.0f}')
axes[1,0].set_xticks(exposure_ticks)
axes[1,0].set_xticklabels(exposure_labels)
axes[1,0].set(title="历史曝光次数", xlabel="曝光次数", ylabel="用户数")
axes[1,0].legend()

# 图3-1d:用户历史接收去重曝光视频数
video_log, video_ticks, video_labels = log1p_axis_values(u["unique_exposed_videos"])
sns.histplot(x=video_log, bins=20, ax=axes[1,1], edgecolor="black")
axes[1,1].axvline(np.log1p(u["unique_exposed_videos"].median()), color="red", linestyle="--", label=f'中位数={u["unique_exposed_videos"].median():.0f}')
axes[1,1].set_xticks(video_ticks)
axes[1,1].set_xticklabels(video_labels)
axes[1,1].set(title="历史去重曝光视频数", xlabel="去重视频数", ylabel="用户数")
axes[1,1].legend()

# 图片输出
fig.suptitle("图3-1 用户历史活跃与曝光规模", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig( OUT_PHASE3 / "P3_01_user_activity_exposure.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 用户历史活跃度较高：活跃天数中位数为8天，最近一次活跃距快照日中位数仅1天。
* 不同用户的历史曝光次数差异较大：历史曝光次数中位数28次，同时存在明显长尾分布。低曝光次数用户计算观看率、互动率时容易受少量行为放大，因此后续需要设置最低曝光次数门槛后再分析行为表现。

### 3.2.2 用户历史观看与互动行为表现（基于曝光次数门槛）
分析基于曝光次数门槛的观看与互动率、以及每万次曝光行为次数。

由于低曝光用户的行为率容易受到少量行为事件影响而产生较大波动，因此采用历史曝光次数的P30作为最低观测门槛，仅对达到该门槛的用户进行分析。

In [ ]:
#（1）设置曝光次数门槛
USER_EXP_THRESHOLD = int(u.loc[u["exposures"] > 0, "exposures"].quantile(0.30))   # 在有曝光用户中计算曝光次数P30，作为行为率分析的最低观测门槛
u_exp = u[u["exposures"] >= USER_EXP_THRESHOLD].copy()  # 取出满足曝光次数门槛的用户数据

print(f"最低曝光门槛={USER_EXP_THRESHOLD}")
print(f"保留用户={len(u_exp)}，占全量用户{len(u_exp)/len(u)*100:.1f}%")
print(f"占有曝光用户{len(u_exp)/(u['exposures'] > 0).sum()*100:.1f}%")

In [ ]:
# （2）用户历史观看、互动行为描述性统计
view_cols = ["valid_view_events","long_view_events","complete_view_events","total_play_time_s",'avg_play_time_per_exposure_s']  # 观看行为规模
interaction_cols = ["likes","comments","forwards","hates","profile_enters","follows"]                                           # 互动行为规模

view_rate_cols = ["valid_view_rate", "long_view_rate", "complete_view_rate"]                                                    # 观看行为率
interaction_rate_cols = ["like_rate", "comment_rate", "forward_rate", "hate_rate", "profile_enter_rate", "follow_rate"]         # 互动行为率

display(u_exp[view_cols].describe().style.format("{:.2f}"))
display(u_exp[interaction_cols].describe().style.format("{:.4f}"))
display(u_exp[view_rate_cols].describe().style.format("{:.4f}"))
display(u_exp[interaction_rate_cols].describe().style.format("{:.6f}"))

In [ ]:
# （3）构建整体每万次曝光互动行为次数（辅助绘图）：由于六类互动行为均值、中位数等指标都较低（中位数基本都为0），该部分采用“每1万次曝光实际产生多少次该行为”指标进行绘图
interaction_per_10k = pd.DataFrame({                                    # 六类互动行为的整体行为次数/每万次曝光
    "行为": ["点赞", "评论", "转发", "点踩", "进入主页", "关注"],
    "每万次曝光行为次数": [
        u_exp["likes"].sum()          / u_exp["exposures"].sum() * 10000,
        u_exp["comments"].sum()       / u_exp["exposures"].sum() * 10000,
        u_exp["forwards"].sum()       / u_exp["exposures"].sum() * 10000,
        u_exp["hates"].sum()          / u_exp["exposures"].sum() * 10000,
        u_exp["profile_enters"].sum() / u_exp["exposures"].sum() * 10000,
        u_exp["follows"].sum()        / u_exp["exposures"].sum() * 10000
    ]})

In [ ]:
## 图3-2：用户历史观看与互动表现
fig = plt.figure(figsize=(14, 10))
gs  = fig.add_gridspec(2, 2)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# 图3-2a：观看行为率
view_plot = u_exp[["valid_view_rate", "long_view_rate", "complete_view_rate"]].rename(columns={"valid_view_rate": "有效观看率", "long_view_rate": "长播放率", "complete_view_rate": "完播率"})
sns.boxplot(data=view_plot, showfliers=False, ax=ax1, palette=["royalblue","orange","purple"])
ax1.set(title="观看行为率分析", xlabel="观看行为率", ylabel="行为率", ylim=(0, 1.05))

# 图3-2b：每次曝光平均播放时长
avg_play_log, avg_play_ticks, avg_play_labels = log1p_axis_values(u_exp["avg_play_time_per_exposure_s"])  # 该字段呈长尾分布，进行log1p坐标转换
sns.histplot(x=avg_play_log, bins=20, ax=ax2, edgecolor="black")
ax2.axvline(np.log1p(u_exp["avg_play_time_per_exposure_s"].median()), color="red", linestyle="--", label=f'中位数={u_exp["avg_play_time_per_exposure_s"].median():.1f}')
ax2.set_xticks(avg_play_ticks)
ax2.set_xticklabels(avg_play_labels)
ax2.set(title="每次曝光平均播放时长", xlabel="秒 / 次曝光", ylabel="用户数")
ax2.legend()

# 图3-2c：六类互动行为每万次曝光行为数
sns.barplot(data=interaction_per_10k, x="行为", y="每万次曝光行为次数", ax=ax3, color="royalblue")
ax3.bar_label(ax3.containers[0], fmt="%.1f", padding=4)
ax3.set(title="互动行为整体表现", xlabel="用户行为", ylabel="每万次曝光行为次数")
ax3.set_ylim(0, interaction_per_10k["每万次曝光行为次数"].max() * 1.15)

fig.suptitle(f"图3-2 用户历史观看与互动表现（基于曝光门槛：曝光次数≥{USER_EXP_THRESHOLD}，用户数量={len(u_exp)}）", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE3 / "P3_02_user_view_interaction.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 以16次历史曝光作为最低观测门槛后，共18329名用户进入正式表现分析，占有曝光用户70.0%，在保留多数用户的同时减少低样本率指标波动；
* 有效观看率、长播放率和完播率中位数依次为48.6%、35.9%和15.0%，随观看要求加深明显下降；
* 互动行为明显比观看行为稀疏，且主要集中在主页进入和点赞。转发、关注和点踩发生频率很低。

### 3.2.3 用户视频标签偏好结构
分析用户的Top1标签分布、偏好丰富度、偏好集中度。

In [ ]:
#（1）用户视频标签偏好画像覆盖
u_positive = u_active[u_active["interest_diversity"] > 0].copy()    # 至少有1个正向偏好标签的用户数
n_interest = len(u_positive)                                        # 至少有1个正向偏好标签的用户数
n_top2 = u_positive["top2_interest_tag"].notna().sum()              # 至少有2个正向偏好标签的用户数

print(f"有正向偏好标签用户={n_interest}，占全量用户{n_interest/len(u)*100:.1f}%，占历史曝光用户{n_interest/len(u_active)*100:.1f}%")
print(f"有至少2个正向偏好标签用户={n_top2}，占正向偏好用户{n_top2/len(u_positive)*100:.1f}%")

# （2）用户Top1 视频标签偏好分布
top1_distribution = (u_positive["top1_interest_tag"].astype(str).value_counts().rename_axis("标签").reset_index(name="用户数"))  # 统计各Top1标签对应的用户数
top1_distribution["用户占比%"] = (top1_distribution["用户数"] / len(u_positive) * 100) # 计算各Top1标签用户数占全部正向偏好用户的比例

display(top1_distribution.head(15))  # 展示Top1偏好用户数最多的前15个视频标签

# （3）用户正向偏好丰富度与集中度
interest_structure_cols = ["interest_diversity","top1_interest_share","top2_interest_share"]    # 用户偏好指标
display(u_positive[interest_structure_cols].describe().style.format("{:.2f}"))                  # 查看正向偏好用户在兴趣丰富度和集中度指标上的描述统计

In [ ]:
## 图3-3：用户视频标签偏好结构
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图3-3a：正向偏好丰富度分布
sns.countplot(data=u_positive, x="interest_diversity", ax=axes[0,0], color="green")
axes[0,0].axvline(u_positive["interest_diversity"].median(), color="red", linestyle="--", label=f'中位数={u_positive["interest_diversity"].median():.0f}')
axes[0,0].set(title="正向偏好丰富度分布", xlabel="正向偏好标签数量", ylabel="用户数")
axes[0,0].legend()

# 图3-3b：Top1偏好集中度分布
sns.histplot(data=u_positive, x="top1_interest_share", bins=15, ax=axes[0,1], edgecolor="black")
axes[0,1].axvline(u_positive["top1_interest_share"].median(), color="red", linestyle="--", label=f'中位数={u_positive["top1_interest_share"].median():.2f}')
axes[0,1].set(title="Top1偏好集中度分布", xlabel="Top1标签得分占全部正向偏好得分比例", ylabel="用户数")
axes[0,1].legend()

# 图3-3c：Top2偏好集中度分布
sns.histplot(data=u_positive, x="top2_interest_share", bins=15, ax=axes[1,0], edgecolor="black")
axes[1,0].axvline(u_positive["top2_interest_share"].median(), color="red", linestyle="--", label=f'中位数={u_positive["top2_interest_share"].median():.2f}')
axes[1,0].set(title="Top2偏好集中度分布", xlabel="Top2标签得分占全部正向偏好得分比例", ylabel="用户数")
axes[1,0].legend()

# 图3-3d：Top1偏好标签前15
t1 = top1_distribution.head(15).sort_values("用户数", ascending=False)
sns.barplot(data=t1, x="用户数", y="标签", ax=axes[1,1], color="royalblue", orient="h")
for i, row in t1.reset_index(drop=True).iterrows():
    axes[1,1].text(row["用户数"], i, f'  {int(row["用户数"])}（{row["用户占比%"]:.1f}%）', va="center", ha="left", fontsize=9)
axes[1,1].set(title="Top1偏好标签前15", xlabel="用户数", ylabel="视频标签")
axes[1,1].set_xlim(0, t1["用户数"].max()*1.25)

fig.suptitle("图3-3 用户视频标签偏好结构", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE3 / "P3_03_user_interest_structure.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 用户偏好画像覆盖率较高：92.8%的全量用户拥有至少1个正向偏好标签，且93.9%的正向偏好用户具有Top2偏好；
* 用户正向偏好标签数中位数为8个，Top1偏好仅占正向偏好总得分的35%，Top2偏好占比中位数为19%，说明多数用户具有多标签偏好，而非集中于单一标签。
* 历史曝光用户中96.66%的净偏好得分大于0，3.30%等于0，仅10名用户（0.0382%）小于0，整体负向净偏好极少；
* 匿名标签39覆盖51.3%的Top1偏好用户，集中度较高。

### 3.2.4 用户历史活跃强度与反馈强度分层
基于历史窗口中的活跃强度与用户反馈强度构建二维行为分层。
1. 首先以历史曝光用户曝光次数的P30作为最低观测门槛，筛选正式分层样本，其余用户保留为低曝光次数或零曝光类型；
2. 使用历史活跃天数与历史曝光次数衡量用户活跃强度，使用观看行为与互动行为衡量用户反馈强度；
3. 分别计算active_score与feedback_score，并采用其P70作为高水平分层切点，将正式分层用户划分为四种类型。

In [ ]:
# 定义赋分函数：将指标先转换为样本内平均秩，再把排名线性缩放至0～1。得分表示用户在当前正式样本中的相对位置。
def rank_score(x:pd.Series):
    rank_val = x.rank(method='average')
    if rank_val.max() == rank_val.min():
        return pd.Series(0.0, index=x.index)
    return (rank_val - rank_val.min()) / (rank_val.max() - rank_val.min())

In [ ]:
# （1）设置正式分层用户样本
u_eligible = u_exp.copy()       # 复制达到曝光门槛的用户数据，进行分层

# （2）为用户活跃、观看与互动表现赋分（赋分0~1）
# 活跃强度
u_eligible["active_days_score"]   = rank_score(u_eligible["active_days"])                    # 活跃天数得分
u_eligible["exposure_score"]      = rank_score(u_eligible["exposures"])                      # 曝光次数得分
# 观看深度
u_eligible['avg_play_time_score'] = rank_score(u_eligible["avg_play_time_per_exposure_s"])   # 次均播放时长得分
u_eligible["valid_view_score"]    = rank_score(u_eligible["valid_view_rate"])                # 有效观看率得分
u_eligible["long_view_score"]     = rank_score(u_eligible["long_view_rate"])                 # 长播放率得分
u_eligible["complete_view_score"] = rank_score(u_eligible["complete_view_rate"])             # 完播率得分
# 互动强度
u_eligible["like_score"]          = rank_score(u_eligible["like_rate"])                      # 点赞率得分
u_eligible['comment_score']       = rank_score(u_eligible['comment_rate'])                   # 评论率得分
u_eligible["forward_score"]       = rank_score(u_eligible["forward_rate"])                   # 转发率得分
u_eligible["profile_enter_score"] = rank_score(u_eligible["profile_enter_rate"])             # 进入主页率得分
u_eligible["follow_score"]        = rank_score(u_eligible["follow_rate"])                    # 关注率得分
u_eligible["hate_score"]          = rank_score(u_eligible["hate_rate"])                      # 点踩率得分

# （3）汇总二维度得分（将得分汇总为 活跃强度得分、反馈强度得分 两个维度）
u_eligible["active_score"] = (                       # 活跃强度得分
      u_eligible["active_days_score"]   * 0.40
    + u_eligible["exposure_score"]      * 0.60)

u_eligible["view_score"] = (                      # 观看深度得分
      u_eligible["avg_play_time_score"] * 0.15
    + u_eligible["valid_view_score"]    * 0.20
    + u_eligible["long_view_score"]     * 0.30
    + u_eligible["complete_view_score"] * 0.35)

u_eligible["interaction_score"] = (               # 互动强度得分 (取等权)
      u_eligible["like_score"]          * 0.1667
    + u_eligible["comment_score"]       * 0.1667
    + u_eligible["forward_score"]       * 0.1667
    + u_eligible["profile_enter_score"] * 0.1667
    + u_eligible["follow_score"]        * 0.1667
    + u_eligible["hate_score"]          * 0.1667)  # 点踩也属于互动事件，因此参与互动强度得分；

# 将观看得分与互动得分汇总为用户反馈强度得分得分
u_eligible['feedback_score'] = (u_eligible["view_score"] + u_eligible["interaction_score"]) /2

print("\n用户活跃强度与反馈强度得分情况：")
display(u_eligible[["active_score","feedback_score"]].describe().round(4))

In [ ]:
# （4）划分活跃强度得分、反馈强度得分P70分层阈值，作为分层高水平切点
USER_ACTIVITY_CUTOFF   = u_eligible["active_score"].quantile(0.70)  # 取出active_score的P70分位数值，作为分层切点
USER_FEEDBACK_CUTOFF = u_eligible["feedback_score"].quantile(0.70)    # 取出interaction_score的P70分位数值，作为分层切点

print(f"用户活跃强度得分P70切点={USER_ACTIVITY_CUTOFF:.2f}")
print(f"用户反馈强度得分P70切点={USER_FEEDBACK_CUTOFF:.2f}")

m_hi_act = u_eligible["active_score"] >= USER_ACTIVITY_CUTOFF       # 参与分层的用户中，达到active_score分层切点的用户
m_hi_eng = u_eligible["feedback_score"] >= USER_FEEDBACK_CUTOFF       # 参与分层的用户中，达到interaction_score分层切点的用户

In [ ]:
# （5）进行用户分层
u_eligible["user_segment"] = "一般活跃一般反馈用户"                                # 设置默认用户类型为“无历史曝光用户”
u_eligible.loc[ m_hi_act &  m_hi_eng, "user_segment"] = "高活跃高反馈用户"
u_eligible.loc[ m_hi_act & ~m_hi_eng, "user_segment"] = "高活跃一般反馈用户"
u_eligible.loc[~m_hi_act &  m_hi_eng, "user_segment"] = "一般活跃高反馈用户"

#（6）将用户分层标签与得分写回全量用户
user_seg = u.copy()     # 取全量用户进行类型划分
                        
user_seg["user_segment"] = "无历史曝光用户"      # 在最终视频类型表，设置默认视频类型
user_seg.loc[(user_seg["exposures"] > 0) & (user_seg["exposures"] < USER_EXP_THRESHOLD),"user_segment"] = "低曝光次数用户"
user_score_cols = ["active_days_score",
                   "exposure_score",
                   'active_score',
                   'avg_play_time_score',
                   'valid_view_score',
                   "long_view_score",
                   "complete_view_score",
                   "view_score",
                   "like_score",
                   'comment_score',
                   'forward_score',
                   'profile_enter_score',
                   'follow_score',
                   'hate_score',
                   "interaction_score",
                   "feedback_score"]

user_seg[user_score_cols] = u_eligible[user_score_cols]                       # 将对应得分写入最终用户类型表
user_seg.loc[u_eligible.index, "user_segment"] = u_eligible["user_segment"]   # 将对应用户类型写入最终用户类型表（只覆盖参与正式分层的用户，对于未参与正式分层的用户不作写入）

# （7）统计各用户类型的用户数量及占全量用户比例，并输出用户历史分层结果
user_seg_order = ["高活跃高反馈用户","高活跃一般反馈用户","一般活跃高反馈用户","一般活跃一般反馈用户","低曝光次数用户","无历史曝光用户"]  # 固定用户标签顺序

user_segment_result = user_seg["user_segment"].value_counts().reindex(user_seg_order).rename("用户数").reset_index() # 各用户类型的用户数量
user_segment_result["占全量用户%"] = user_segment_result["用户数"] / len(user_seg) * 100 # 各用户类型的用户数占全量用户数比例
display(user_segment_result.round(2))

### 3.2.5 用户类型画像分析
从基础属性、身份特征、近期活跃状态与 视频标签偏好等未参与分层的字段，对六类用户进行补充画像分析

In [ ]:
# （1）构建用户类型综合画像汇总表
user_seg_summary = user_seg.groupby("user_segment", dropna=False).agg(
    用户数=("user_id","count"),                                              # 各类型的用户数量
    # 用户基础属性
    注册天数中位数 =("register_days", "median"),
    粉丝数中位数 =("fans_user_num", "median"),
    关注用户数中位数 =("follow_user_num", "median"),
    好友数中位数=("friend_user_num", "median"),
    作者人数占用户比例=("is_video_author", "mean"),
    直播人数占用户比例=("is_live_streamer", "mean"),
    # 近期活跃状态
    最近活跃间隔中位数=("days_since_last_active", "median"),
    # 曝光规模
    去重曝光视频数中位数=("unique_exposed_videos","median"),
    总播放时长中位数=('total_play_time_s','median'),
    #  视频标签偏好
    正向偏好画像覆盖率=("interest_diversity",lambda x: (x > 0).mean()),
    偏好标签数中位数=("interest_diversity", "median"),
    Top1偏好得分中位数=("top1_interest_score", "median"),
    Top2偏好得分中位数=("top2_interest_score", "median"),
    正向偏好总得分中位数=("total_positive_interest_score", "median"),
    Top1偏好集中度中位数=("top1_interest_share", "median"),
    Top2偏好集中度中位数=("top2_interest_share", "median"),
    净偏好得分中位数=("net_interest_score", "median"),
    净偏好为负用户占比=("net_interest_score", lambda x: (x.dropna() < 0).mean() if x.notna().any() else np.nan)
).reindex(user_seg_order)

# 计算各类型用户占全量用户的比例
user_seg_summary["占全部用户%"] = user_seg_summary["用户数"] / len(user_seg) * 100

# 将比例字段转换为百分数
percent_cols = ["作者人数占用户比例","直播人数占用户比例","正向偏好画像覆盖率",'Top1偏好集中度中位数','Top2偏好集中度中位数','净偏好为负用户占比']
user_seg_summary[percent_cols] = user_seg_summary[percent_cols] * 100

# 整理为最终展示表并输出
user_seg_table = user_seg_summary.reset_index()
display(user_seg_table.round(4))
user_seg_table.to_excel(OUT_PHASE3 / "user_seg_table.xlsx", index=False)

In [ ]:
# （2）不同用户类型的基础画像特征分析（对各用户类型的人数占比、注册天数等基础属性进行分析）
# 整理绘图数据
segment_plot = user_seg_summary.reset_index()

# 把"作者占比"和"直播占比"两列堆叠成一列
identity_plot = user_seg_summary[["作者人数占用户比例", "直播人数占用户比例"]].reset_index().melt(
    id_vars="user_segment",
    var_name="指标",
    value_name="占比"
)

register_plot = user_seg_summary[["注册天数中位数"]].reset_index()

recency_plot = user_seg_summary[["最近活跃间隔中位数"]].reset_index().dropna()   # 无历史曝光用户通常没有有效的最近活跃间隔，因此仅展示有历史曝光的用户类型
recency_order = recency_plot["user_segment"].tolist()

In [ ]:
## 图3-4：不同用户类型的基础画像特征
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 图3-4a：用户类型与人数占比
sns.barplot(data=segment_plot, x="用户数", y="user_segment", order=user_seg_order, ax=axes[0,0], color="royalblue")
for i, row in segment_plot.iterrows():
    axes[0,0].text(row["用户数"], i, f'{int(row["用户数"])} ({row["占全部用户%"]:.1f}%)', va="center", fontsize=9)
axes[0,0].set(title="用户类型数量与占比", xlabel="用户数", ylabel="")
axes[0,0].set_xlim(0, segment_plot["用户数"].max() * 1.22)

# 图3-4b：最近活跃间隔
sns.barplot(data=recency_plot, x="user_segment", y="最近活跃间隔中位数", order=recency_order, ax=axes[0,1], color="orange")
axes[0,1].bar_label(axes[0,1].containers[0], fmt="%.0f", padding=3)
axes[0,1].set(title="最近活跃间隔", xlabel="", ylabel="距最近一次活跃天数中位数")
axes[0,1].tick_params(axis="x", rotation=20)

# 图3-4c：用户身份属性
sns.barplot(data=identity_plot, x="user_segment", y="占比", hue="指标", order=user_seg_order, ax=axes[1,0])
for container in axes[1,0].containers:
    axes[1,0].bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)
axes[1,0].set(title="用户身份属性", xlabel="", ylabel="用户占比（%）")
axes[1,0].tick_params(axis="x", rotation=20)

# 图3-4d：注册天数中位数
sns.barplot(data=register_plot, x="user_segment", y="注册天数中位数", order=user_seg_order, ax=axes[1,1], color="green")
axes[1,1].bar_label(axes[1,1].containers[0], fmt="%.0f", padding=3)
axes[1,1].set(title="用户注册时长", xlabel="", ylabel="注册天数中位数（天）")
axes[1,1].tick_params(axis="x", rotation=20)

fig.suptitle("图3-4 不同用户类型的基础画像特征", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE3 / "P3_04_user_profile.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# （3）不同用户类型的 视频标签偏好分析
preference_user_order = ["高活跃高反馈用户","高活跃一般反馈用户","一般活跃高反馈用户","一般活跃一般反馈用户","低曝光次数用户"]     #  视频标签偏好只保留有历史活跃数据的用户
preference_plot = user_seg_summary[["正向偏好画像覆盖率","偏好标签数中位数","Top1偏好集中度中位数","Top2偏好集中度中位数"]].reset_index()
concentration_plot = preference_plot[["user_segment","Top1偏好集中度中位数","Top2偏好集中度中位数"]].melt(id_vars="user_segment", var_name="指标", value_name="占比")

## 图3-5：不同用户类型的视频标签偏好特征
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 图3-5a：正向偏好画像覆盖率
sns.barplot(data=preference_plot, x="user_segment", y="正向偏好画像覆盖率", order=preference_user_order, ax=axes[0], color="royalblue")
axes[0].bar_label(axes[0].containers[0], fmt="%.1f%%", padding=3)
axes[0].set(title="正向偏好画像覆盖率", xlabel="", ylabel="用户占比（%）")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_ylim(0, 110)

# 图3-5b：正向偏好丰富度
sns.barplot(data=preference_plot, x="user_segment", y="偏好标签数中位数", order=preference_user_order, ax=axes[1], color="orange")
axes[1].bar_label(axes[1].containers[0], fmt="%.0f", padding=3)
axes[1].set(title="正向偏好丰富度", xlabel="", ylabel="正向偏好标签数中位数")
axes[1].tick_params(axis="x", rotation=20)
axes[1].set_ylim(0, preference_plot["偏好标签数中位数"].max() * 1.15)

# 图3-5c：Top1 / Top2偏好集中度
sns.barplot(data=concentration_plot, x="user_segment", y="占比", hue="指标", order=preference_user_order, ax=axes[2])
for container in axes[2].containers: axes[2].bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)
axes[2].set(title="Top1 / Top2偏好集中度", xlabel="", ylabel="占正向偏好总得分比例（%）")
axes[2].tick_params(axis="x", rotation=20)

fig.suptitle("图3-5 不同用户类型的视频标签偏好特征", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE3 / "P3_05_user_type_preference.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 六类用户中，一般活跃一般反馈用户和低曝光次数用户占比最高（31.9%），无历史曝光用户占比最低（4.1%），高活跃高反馈用户占比也较低（5.0%）；
* 高活跃用户最近活跃间隔中位数为0天，一般活跃用户约为1天，低曝光次数用户为4天，不同历史用户类型在近期活跃程度上有明显差异；
* 高活跃高反馈用户的正向偏好标签数中位数为17个、Top1集中度为25.6%；低曝光次数用户正向偏好标签数中位数为3个、Top1集中度为50.0%。因此，当前日志中可观测到的高活跃高反馈用户视频标签偏好更分散，而低曝光次数用户的可观测视频标签偏好更集中。

# Phase 4. 视频与作者历史表现分析
Phase 3 已经从需求侧建立了用户画像和用户行为分层，回答了"用户是谁、怎么使用平台"。
Phase 4 转向供给侧，将视频和作者放在同一章，因为二者都描述"平台向用户展示了哪些视频、作者作品获得了多少曝光"：

## 4.1 SQL:构建视频/作者历史表现特征表
使用画像快照日前的历史数据，构建一套“一行一个视频/作者作品”的历史画像

本阶段构建的供后续使用的表：
* ads_video_pre_performance_feature:视频历史表现特征表，汇总视频在standard_pre窗口内的视频供给曝光、观看和互动等历史表现
* ads_author_pre_performance_feature：作者历史表现特征表，汇总作者在standard_pre窗口内的作品供给、曝光、观看和互动等历史表现

## 4.2 视频历史表现分析

In [ ]:
# （1）先加载数据
video_pre_feature  = pd.read_sql("SELECT * FROM ads_video_pre_performance_feature",  con=engine)
bridge_video_tag = pd.read_sql("SELECT * FROM bridge_video_tag", con=engine)   # 读取视频—标签桥接表

# （2）构建表格分析副本，方便后续操作
v = video_pre_feature.copy()

# （3）转换时间单位
v["duration_min"] = v["video_duration_s"] / 60.0

### 4.2.1 视频供给与历史曝光覆盖

In [ ]:
# （1）视频供给与曝光规模描述统计
n_video         = len(v)                                   # 全量视频数
n_exp_video     = (v["exposures"] > 0).sum()               # 存在曝光行为的视频数
n_no_exp_video  = n_video-n_exp_video                       # 无历史曝光视频数
n_tag_video     = (v["tag_count"] > 0).sum()               # 存在标签的视频数
n_one_tag       = (v["tag_count"] == 1).sum()              # 标签数量为1的视频数
n_multi_tag     = (v["tag_count"] >= 2).sum()              # 标签数量>=2的视频数

print(f"全量视频数={n_video}")
print(f"有历史曝光视频数={n_exp_video}，占全量视频{n_exp_video/n_video*100:.1f}%")
print(f"存在标签视频数={n_tag_video}，占全量视频{n_tag_video/n_video*100:.1f}%")
print(f"单标签视频数={n_one_tag}，占全量视频{n_one_tag/n_video*100:.1f}%")
print(f"多标签视频数={n_multi_tag}，占全量视频{n_multi_tag/n_video*100:.1f}%")

display(v[["video_duration_s","tag_count","exposures","unique_exposed_users"]].describe().style.format("{:.2f}"))

In [ ]:
# （2）整理绘图数据
tag_supply = bridge_video_tag[bridge_video_tag["video_id"].isin(v["video_id"])]["tag"].value_counts().head(15).rename_axis("tag").reset_index(name="视频数")       # 拿到出现频率最高的前15个tag
tag_supply["tag"] = tag_supply["tag"].astype(str)    # 将tag字段转字符串
tag_supply["占有标签视频%"] = tag_supply["视频数"] / n_tag_video * 100


## 图4-1：视频供给与历史曝光覆盖
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

## 图4-1a：视频历史曝光覆盖
sns.barplot(x=["有曝光", "零曝光"], y=[n_exp_video, n_no_exp_video], ax=axes[0,0], edgecolor="black")
axes[0,0].text(0, n_exp_video,    f'视频数：{n_exp_video}\n占全量视频：{n_exp_video/n_video*100:.1f}%',    ha="center", va="bottom", fontsize=9)
axes[0,0].text(1, n_no_exp_video, f'视频数：{n_no_exp_video}\n占全量视频：{n_no_exp_video/n_video*100:.1f}%', ha="center", va="bottom", fontsize=9)
axes[0,0].set(title="视频历史曝光覆盖", xlabel="", ylabel="视频数")
axes[0,0].set_ylim(0, max(n_exp_video, n_no_exp_video)*1.15)

## 图4-1b:视频标签数量分析
sns.barplot(x=["0个标签", "1个标签", "2个及以上"], y=[n_video-n_tag_video, n_one_tag, n_multi_tag], ax=axes[0,1], edgecolor="black")
axes[0,1].text(0, n_video-n_tag_video, f'视频数：{n_video-n_tag_video}\n占全量视频：{(n_video-n_tag_video)/n_video*100:.1f}%', ha="center", va="bottom", fontsize=9)
axes[0,1].text(1, n_one_tag,     f'视频数：{n_one_tag}\n占全量视频：{n_one_tag/n_video*100:.1f}%',     ha="center", va="bottom", fontsize=9)
axes[0,1].text(2, n_multi_tag,   f'视频数：{n_multi_tag}\n占全量视频：{n_multi_tag/n_video*100:.1f}%', ha="center", va="bottom", fontsize=9)
axes[0,1].set(title="视频标签数量分析", xlabel="", ylabel="视频数")
axes[0,1].set_ylim(0, max(n_video-n_tag_video, n_one_tag, n_multi_tag)*1.15)

## 图4-1c：视频标签供给分布（Top15）
tag_supply_plot = tag_supply.sort_values("视频数", ascending=False)   # 降序排序，seaborn 横向图第一行在底部，视觉上从上到下即从小到大
sns.barplot(data=tag_supply_plot, x="视频数", y="tag", ax=axes[1,0], color="royalblue", orient="h")
for i, row in tag_supply_plot.reset_index(drop=True).iterrows(): axes[1,0].text(row["视频数"],i,f'  {int(row["视频数"])}（{row["占有标签视频%"]:.1f}%）',va="center",ha="left",fontsize=9)
axes[1,0].set(title="视频标签供给分布（Top15）", xlabel="覆盖视频数", ylabel="匿名标签")
axes[1,0].set_xlim(0, tag_supply_plot["视频数"].max()*1.30)

## 图4-1d：视频时长分布
duration_log, duration_ticks, duration_labels = log1p_axis_values(v["duration_min"])   # 该字段呈长尾分布，进行log1p转换以减弱极端值对横轴展示的影响
sns.histplot(x=duration_log, bins=20, ax=axes[1,1], edgecolor="black")
axes[1,1].axvline(np.log1p(v["duration_min"].median()), color="red", linestyle="--", label=f'中位数={v["duration_min"].median():.2f}分钟')
axes[1,1].set_xticks(duration_ticks)
axes[1,1].set_xticklabels(duration_labels)
axes[1,1].set(title="视频时长分布", xlabel="视频时长（分钟）", ylabel="视频数")
axes[1,1].legend()

fig.suptitle("图4-1 视频供给与历史曝光覆盖", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_01_video_supply_exposure.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

结论：
* 视频历史曝光和视频标签覆盖率较高：全量视频中96.3%获得过历史曝光，98.7%具有标签；
* 标签结构分布较稀疏：全量视频75.1%的视频只有1个标签，且匿名标签39出现最频繁；
* 视频时长中位数为1.35分钟，整体以短视频为主。

### 4.2.2 视频历史曝光集中度
分析全量视频在历史窗口获得的曝光次数是否均衡

In [ ]:
# （1）定义Gini函数：判断数据的分散程度，0表示分配完全均匀，数值越接近1表示分布越集中
def gini(data):
    values = pd.to_numeric(pd.Series(data),errors="coerce").dropna().to_numpy(dtype=float)
    values = values[values >= 0]
    if len(values) == 0 or values.sum() == 0:
        return np.nan
    values = np.sort(values)
    n_video = len(values)
    return (2*np.sum(np.arange(1,n_video+1)*values)-(n_video+1)*values.sum())/(n_video*values.sum())

# （2）计算头部视频曝光份额
video_exp_sorted = v["exposures"].sort_values(ascending=False)      # 对全量视频，将曝光次数按大小排序
video_exp_total  = video_exp_sorted.sum()                           # 把所有视频的曝光次数之和
top_rows = []
for pct in [0.01,0.05,0.10,0.20]:   # 取出前0.01,0.05,0.10,0.20的视频，并将其做成表格
    n_top = max(1,int(np.ceil(len(video_exp_sorted)*pct)))
    top_rows.append([f"Top {pct*100:.0f}%",n_top,video_exp_sorted.head(n_top).sum()/video_exp_total])
top_video_exposure = pd.DataFrame(top_rows,columns=["分组","视频数","曝光份额"])

gini_video = gini(video_exp_sorted)     # 前期历史窗口内存在曝光的视频的曝光次数基尼系数

display(top_video_exposure.style.format({"曝光份额":"{:.1%}"}))
print(f"全量视频视频Gini={gini_video:.3f}")

In [ ]:
## 图4-2：视频历史曝光集中度
fig, axes = plt.subplots(1,2,figsize=(14,5.5))

## 图4-2a：头部视频曝光份额
sns.barplot(data=top_video_exposure,x="分组",y=top_video_exposure["曝光份额"]*100,ax=axes[0],edgecolor="black")
for i,row in top_video_exposure.iterrows(): axes[0].text(i,row["曝光份额"]*100,f'曝光份额：{row["曝光份额"]*100:.1f}%\n视频数：{int(row["视频数"])}',ha="center",va="bottom",fontsize=9)
axes[0].set(title="头部视频曝光份额",xlabel="",ylabel="曝光份额（%）")
axes[0].set_ylim(0,top_video_exposure["曝光份额"].max()*100*1.20)

## 图4-2b：视频曝光洛伦兹曲线
x = np.sort(video_exp_sorted.to_numpy())
cum = np.insert(np.cumsum(x),0,0)/np.sum(x)
axes[1].plot(np.linspace(0,1,len(cum)),cum,label=f"全量视频（Gini={gini_video:.3f}）")
axes[1].plot([0,1],[0,1],linestyle="--",label="完全均匀分配")
axes[1].set(title="视频曝光洛伦兹曲线",xlabel="视频累计占比",ylabel="曝光累计占比",xlim=(0,1),ylim=(0,1))
axes[1].legend()

fig.suptitle("图4-2 视频历史曝光集中度",fontsize=18,y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_02_video_exposure_concentration.png",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 视频流量明显向头部集中：全量视频中，Top 20%视频获得76.5%的历史曝光，Gini系数为0.729

### 4.2.3 视频历史观看与互动行为表现（基于曝光次数门槛）
对于有足够曝光量的视频，分析其观看和互动表现

In [ ]:
# （1）设置视频最低曝光观测门槛
VIDEO_EXP_THRESHOLD = int(v.loc[v["exposures"] > 0,"exposures"].quantile(0.30))   # 视频曝光次数P30门槛
v_exp = v[v["exposures"] >= VIDEO_EXP_THRESHOLD].copy() # 取出满足曝光次数门槛的视频数据

print(f"最低曝光门槛={VIDEO_EXP_THRESHOLD}")
print(f"保留视频={len(v_exp)}，占全部视频{len(v_exp)/n_video*100:.1f}%")
print(f"占有曝光视频{len(v_exp)/(v['exposures']>0).sum()*100:.1f}%")

# （2）观看与互动行为表现描述统计
display(v_exp[view_cols].describe().style.format("{:.2f}"))
display(v_exp[view_rate_cols].describe().style.format("{:.4f}"))
display(v_exp[interaction_cols].describe().style.format("{:.4f}"))
display(v_exp[interaction_rate_cols].describe().style.format("{:.6f}"))

In [ ]:
# （3）计算视频整体每万次曝光产生互动行为次数
video_interaction_per_10k = pd.DataFrame({
    "行为": ["点赞","评论","转发","点踩","进入主页","关注"],
    "每万次曝光行为次数": [
        v_exp["likes"].sum()          / v_exp["exposures"].sum()*10000,
        v_exp["comments"].sum()       / v_exp["exposures"].sum()*10000,
        v_exp["forwards"].sum()       / v_exp["exposures"].sum()*10000,
        v_exp["hates"].sum()          / v_exp["exposures"].sum()*10000,
        v_exp["profile_enters"].sum() / v_exp["exposures"].sum()*10000,
        v_exp["follows"].sum()        / v_exp["exposures"].sum()*10000
    ]
})

In [ ]:
## 图4-3：视频历史观看与互动表现
fig = plt.figure(figsize=(14,10))
gs = fig.add_gridspec(2,2)
ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[1,:])

## 图4-3a：视频观看行为率
video_view_plot = v_exp[["valid_view_rate","long_view_rate","complete_view_rate"]].rename(columns={"valid_view_rate":"有效观看率","long_view_rate":"长播放率","complete_view_rate":"完播率"})
sns.boxplot(data=video_view_plot, showfliers=False, ax=ax1)
ax1.set(title="视频观看行为率", xlabel="", ylabel="行为率", ylim=(0,1.05))

## 4-3b：每次曝光平均播放时长分布
video_avg_play_log, video_avg_play_ticks, video_avg_play_labels = log1p_axis_values(v_exp["avg_play_time_per_exposure_s"])
sns.histplot(x=video_avg_play_log, bins=20, ax=ax2, edgecolor="black")
ax2.axvline(np.log1p(v_exp["avg_play_time_per_exposure_s"].median()),color="red",linestyle="--",label=f'中位数={v_exp["avg_play_time_per_exposure_s"].median():.1f}')
ax2.set_xticks(video_avg_play_ticks)
ax2.set_xticklabels(video_avg_play_labels)
ax2.set(title="每次曝光平均播放时长", xlabel="秒 / 次曝光", ylabel="视频数")
ax2.legend()

## 4-3c：视频互动行为整体表现
sns.barplot(data=video_interaction_per_10k, x="行为", y="每万次曝光行为次数", ax=ax3, edgecolor="black")
ax3.bar_label(ax3.containers[0],fmt="%.1f",padding=4)
ax3.set(title="视频互动行为整体表现", xlabel="", ylabel="每万次曝光行为次数（次）")
ax3.set_ylim(0, video_interaction_per_10k["每万次曝光行为次数"].max()*1.15)

fig.suptitle(f"图4-3 视频历史观看与互动表现（曝光次数≥{VIDEO_EXP_THRESHOLD}，视频数量={len(v_exp)}）", fontsize=18, y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_03_video_view_interaction.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 达到最低曝光门槛的视频中，有效观看率、长播放率和完播率中位数依次为39.7%、27.6%和8.6%，观看要求越深，达到的比例越低；
* 互动行为主要集中在主页进入和点赞，评论及其余行为明显更少。

### 4.2.4 视频历史曝光规模与综合表现分层
基于历史窗口中的历史曝光规模与综合表现构建二维视频分层。
1. 首先以有历史曝光视频曝光次数的P30作为最低观测门槛，筛选正式分层样本，其余视频保留为低样本或零曝光类型；
2. 使用历史曝光次数衡量视频曝光规模，使用观看表现与互动表现共同衡量视频获得曝光后的视频历史表现；
3. 以exposures与performance_score的P70作为高水平切点，将正式分层视频划分为四种类型。

In [ ]:
# （1）设置正式分层视频样本
v_eligible = v_exp.copy()    # 复制达到曝光门槛的视频数据，进行分层

# （2）为观看与互动表现赋分
v_eligible['avg_play_time_score'] = rank_score(v_eligible["avg_play_time_per_exposure_s"])  #rank_score函数已在3.2.4用户分层中定义
v_eligible["valid_view_score"]    = rank_score(v_eligible["valid_view_rate"])
v_eligible["long_view_score"]     = rank_score(v_eligible["long_view_rate"])
v_eligible["complete_view_score"] = rank_score(v_eligible["complete_view_rate"])
v_eligible["like_score"]          = rank_score(v_eligible["like_rate"])
v_eligible["forward_score"]       = rank_score(v_eligible["forward_rate"])
v_eligible["profile_enter_score"] = rank_score(v_eligible["profile_enter_rate"])
v_eligible["follow_score"]        = rank_score(v_eligible["follow_rate"])
v_eligible["hate_score"]          = rank_score(v_eligible["hate_rate"])

# （3）观看、互动行为得分（权重根据行为深度和业务解释进行经验赋权。评论的情感方向难以仅根据行为事件判断，因此保留在描述分析中，但不纳入互动得分；转发具有传播价值，但也可能是吐槽、负向反馈，因此给予较低权重。）
v_eligible["view_score"] = (
                              v_eligible["avg_play_time_score"] * 0.15  #次均播放时长得分
                            + v_eligible["valid_view_score"]    * 0.20  #有效观看得分
                            + v_eligible["long_view_score"]     * 0.30  #长播放得分
                            + v_eligible["complete_view_score"] * 0.35  #完整播放得分
                            )
v_eligible["interaction_score"] = (
                              v_eligible["like_score"]          * 0.15   #点赞行为得分
                            + v_eligible["forward_score"]       * 0.10   #转发行为得分
                            + v_eligible["profile_enter_score"] * 0.30   #进入主页行为得分
                            + v_eligible["follow_score"]        * 0.45   #关注行为得分
                            - v_eligible["hate_score"]          * 0.30   #点踩行为得分（负值）
                            )
v_eligible["interaction_score"] = rank_score(v_eligible["interaction_score"])   # 因为原始interaction_score的得分范围是(-0.3~1)
v_eligible["performance_score"] = (v_eligible["view_score"] + v_eligible["interaction_score"]) / 2   # 视频综合表现得分：观看表现与互动表现等权

# (4) 划分曝光次数、表现得分P70分层阈值：采用正式分层视频曝光次数P70和综合表现得分P70作为分层切点
VIDEO_EXPOSURE_CUTOFF = v_eligible["exposures"].quantile(0.70)              # 参与分层的视频的曝光次数P70数值
VIDEO_PERFORMANCE_CUTOFF = v_eligible["performance_score"].quantile(0.70)   # 视频得分的P70数值

print(f"视频曝光规模P70分层切点={VIDEO_EXPOSURE_CUTOFF:.0f}")
print(f"视频综合表现P70分层切点={VIDEO_PERFORMANCE_CUTOFF:.4f}")

m_hi_exp = v_eligible["exposures"] >= VIDEO_EXPOSURE_CUTOFF                 # 达到曝光次数高水平切点
m_hi_perf = v_eligible["performance_score"] >= VIDEO_PERFORMANCE_CUTOFF     # 达到表现分数高水平切点

# （5）进行视频分层
v_eligible["video_segment"] = "一般曝光一般表现视频"                            # 设置默认视频类型为“无历史曝光视频”
v_eligible.loc[m_hi_exp & m_hi_perf,"video_segment"] = "高曝光高表现视频"
v_eligible.loc[m_hi_exp & ~m_hi_perf,"video_segment"] = "高曝光一般表现视频"
v_eligible.loc[~m_hi_exp & m_hi_perf,"video_segment"] = "一般曝光高表现视频"

# （6）将视频分层标签与得分写回全量视频
video_seg = v.copy()                            # 取全量视频进行类型划分
video_seg["video_segment"] = "无历史曝光视频"      # 在最终视频类型表，设置默认视频类型
video_seg.loc[(video_seg["exposures"] > 0) & (video_seg["exposures"] < VIDEO_EXP_THRESHOLD),"video_segment"] = "低曝光次数视频"
video_score_cols = ["avg_play_time_score",
                    "valid_view_score",
                    "long_view_score",
                    "complete_view_score",
                    "like_score",
                    "forward_score",
                    "profile_enter_score",
                    "follow_score",
                    "hate_score",
                    "view_score",
                    "interaction_score",
                    "performance_score"]

video_seg[video_score_cols] = v_eligible[video_score_cols]                       # 将对应得分写入最终视频类型表
video_seg.loc[v_eligible.index, "video_segment"] = v_eligible["video_segment"]

# （7）统计各视频类型的视频数量及占全部视频比例，并输出视频历史分层结果
video_seg_order = ["高曝光高表现视频","高曝光一般表现视频","一般曝光高表现视频","一般曝光一般表现视频","低曝光次数视频","无历史曝光视频"]   # 固定视频类型字段顺序

video_segment_result = video_seg["video_segment"].value_counts().reindex(video_seg_order).rename("视频数").reset_index() # 各视频类型的视频数量
video_segment_result["占全部视频%"] = video_segment_result["视频数"] / len(video_seg) * 100 # 各视频类型的视频数占全部视频数比例
display(video_segment_result.round(2))

In [ ]:
## 图4-4：视频历史分层结果
fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(data=video_segment_result,x="视频数",y="video_segment",order=video_seg_order,ax=ax,edgecolor="black")
for i,row in video_segment_result.set_index("video_segment").reindex(video_seg_order).reset_index().iterrows():
    ax.text(row["视频数"],i,f'  {int(row["视频数"])} ({row["占全部视频%"]:.1f}%)',va="center",fontsize=9)
ax.set(title="图4-4 视频历史分层结果",xlabel="视频数",ylabel="")
ax.set_xlim(0,video_segment_result["视频数"].max()*1.20)

fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_04_video_segmentation.png",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

此处不再分析“不同视频类型基础特征”。
* 一方面，曝光规模及核心观看、互动指标已经直接参与视频分层，继续使用这些指标解释分层结果会形成循环解释；
* 另一方面，剩余较独立的视频时长和标签分布已经在4.2.1进行了分析，当前缺少能够提供明显新增信息的独立视频属性，因此不再为了结构完整重复开展分层后画像。

## 4.3 作者历史表现分析

In [ ]:
# （1）先加载数据
author_pre_feature = pd.read_sql("SELECT * FROM ads_author_pre_performance_feature", con=engine)

# （2）构建表格分析副本，方便后续操作
a = author_pre_feature.copy()

### 4.3.1 作者作品供给与历史曝光覆盖

In [ ]:
# （1）作者作品供给与曝光规模描述统计
n_exposed_author = (a["exposures"] > 0).sum()   # 作品在前期历史窗口得到曝光的作者人数

print(f"全量作者数={len(a)}")
print(f"有历史曝光作者数={n_exposed_author}（{n_exposed_author/len(a)*100:.1f}%）")

display(a[["uploaded_video_count","unique_exposed_videos","exposures","unique_exposed_users"]].describe().style.format("{:.2f}"))

In [ ]:
# （2）整理绘图数据
uploaded_group = a["uploaded_video_count"].apply(lambda x:"≥4个" if x>=4 else f"{int(x)}个")
author_upload_plot = uploaded_group.value_counts().reindex(["1个","2个","3个","≥4个"],fill_value=0).rename_axis("上传视频数").reset_index(name="作者数")
author_upload_plot["占比%"] = author_upload_plot["作者数"]/len(a)*100


## 图4-5：作者作品供给与历史曝光覆盖
fig, axes = plt.subplots(2,2,figsize=(14,10))

## 图4-5a：历史曝光作者覆盖
sns.barplot(x=["有曝光作者", "无历史曝光作者"], y=[n_exposed_author, len(a)-n_exposed_author], ax=axes[0,0], edgecolor="black")
axes[0,0].text(0, n_exposed_author,        f'作者人数：{n_exposed_author}\n占全量作者比例：{n_exposed_author/len(a)*100:.2f}%',        ha="center", va="bottom", fontsize=9)
axes[0,0].text(1, len(a)-n_exposed_author, f'作者人数：{len(a)-n_exposed_author}\n占全量作者比例：{(len(a)-n_exposed_author)/len(a)*100:.2f}%', ha="center", va="bottom", fontsize=9)
axes[0,0].set(title="历史曝光作者覆盖", xlabel="", ylabel="作者数")
axes[0,0].set_ylim(0, max(n_exposed_author, len(a)-n_exposed_author)*1.15)

## 图4-5b：每个作者作品供给规模
sns.barplot(data=author_upload_plot,x="上传视频数",y="作者数",ax=axes[0,1],edgecolor="black")
for i,row in author_upload_plot.iterrows(): axes[0,1].text(i,row["作者数"],f'作者人数：{int(row["作者数"])}\n占全量作者比例：{row["占比%"]:.2f}%',ha="center",va="bottom",fontsize=9)
axes[0,1].set(title="每个作者作品供给规模",xlabel="快照日前上传视频数",ylabel="作者数")
axes[0,1].set_ylim(0,author_upload_plot["作者数"].max()*1.15)

#图4-5c：作者作品历史曝光规模
author_exp_log, author_exp_ticks, author_exp_labels = log1p_axis_values(a["exposures"])
sns.histplot(x=author_exp_log,bins=25,ax=axes[1,0],edgecolor="black")
axes[1,0].axvline(np.log1p(a["exposures"].median()),color="red",linestyle="--",label=f'中位数={a["exposures"].median():.0f}')
axes[1,0].set_xticks(author_exp_ticks)
axes[1,0].set_xticklabels(author_exp_labels)
axes[1,0].set(title="作者作品历史曝光规模",xlabel="作品曝光次数",ylabel="作者数")
axes[1,0].legend()

## 图4-5d：作者去重用户触达规模
author_reach_log, author_reach_ticks, author_reach_labels = log1p_axis_values(a["unique_exposed_users"])
sns.histplot(x=author_reach_log,bins=25,ax=axes[1,1],edgecolor="black")
axes[1,1].axvline(np.log1p(a["unique_exposed_users"].median()),color="red",linestyle="--",label=f'中位数={a["unique_exposed_users"].median():.0f}')
axes[1,1].set_xticks(author_reach_ticks)
axes[1,1].set_xticklabels(author_reach_labels)
axes[1,1].set(title="作者去重用户触达规模",xlabel="去重触达用户数",ylabel="作者数")
axes[1,1].legend()

fig.suptitle("图4-5 作者作品供给与历史曝光覆盖",fontsize=18,y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_05_author_supply_exposure.png",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 96.2%的作者作品获得过历史曝光，87.0%的作者只上传1个视频；
* 作者总曝光次数和触达用户数呈明显长尾分布，说明作者之间差异主要表现在作品获得的流量而不是作品发布数量；

### 4.3.2 作者作品供给与历史曝光集中度

In [ ]:
# （1）计算头部作者上传视频份额
author_supply_video_sorted = a["uploaded_video_count"].sort_values(ascending=False)     # 对全量作者，按作者上传作品数降序排列
author_supply_video_total = author_supply_video_sorted.sum()                            # 全体作者的上传视频总数

top_author_video = []
for pct in [0.01,0.05,0.10,0.20]:
    n_top = max(1,int(len(author_supply_video_sorted)*pct))                               # 该组作者人数
    video_share = (author_supply_video_sorted.head(n_top).sum() / author_supply_video_total)    # 该组作者上传的作品数占全量作品的比例
    top_author_video.append([
        f"Top {pct*100:.0f}%",                                                      # 上传作品数在前pct的作者组
        n_top,
        author_supply_video_sorted.head(n_top).sum()/author_supply_video_total])

top_author_video_supply = pd.DataFrame(top_author_video,columns=["分组","作者数","上传视频份额"])

gini_author_video_supply = gini(a["uploaded_video_count"])                           # 作者上传视频数的基尼系数

display(top_author_video_supply.style.format({"上传视频份额":"{:.2%}"}))
print(f"全量作者上传视频数Gini={gini_author_video_supply:.3f}")

In [ ]:
# （2）计算头部作者作品曝光份额
author_exposure_sorted = (a["exposures"].sort_values(ascending=False))      # 按作者作品历史曝光次数降序排列
author_exposure_total = author_exposure_sorted.sum()                        # 全体作者作品历史曝光总次数

top_author_exposure = []

for pct in [0.01, 0.05, 0.10, 0.20]:
    n_top = max(1,int(len(author_exposure_sorted) * pct))                                # 该组作者人数
    exposure_share = (author_exposure_sorted.head(n_top).sum() / author_exposure_total)  # 当前头部作者曝光次数占全部作者曝光次数的比例
    top_author_exposure.append([
        f"Top {pct * 100:.0f}%",
        n_top,
        exposure_share
    ])

top_author_exposure = pd.DataFrame(top_author_exposure,columns=["分组", "作者数", "曝光份额"])

gini_author_exposure = gini(a["exposures"])         # 计算作者作品历史曝光次数的基尼系数

display(top_author_exposure.style.format({"曝光份额": "{:.2%}"}))

print(f"全量作者作品曝光次数Gini={gini_author_exposure:.3f}")

In [ ]:
## 图4-6：作者作品供给与历史曝光集中度
fig, axes = plt.subplots(2,2,figsize=(14,10))

# 图4-6a：头部作者上传视频份额
sns.barplot(data=top_author_video_supply,x="分组",y=top_author_video_supply["上传视频份额"] * 100,ax=axes[0, 0],edgecolor="black")
for i, row in top_author_video_supply.iterrows():
    axes[0, 0].text(i,row["上传视频份额"] * 100,f'{row["上传视频份额"] * 100:.1f}%\n作者数：{int(row["作者数"])}',ha="center",va="bottom",fontsize=9)
axes[0, 0].set(title="头部作者上传视频份额",xlabel="",ylabel="上传视频份额（%）")
axes[0, 0].set_ylim(0,top_author_video_supply["上传视频份额"].max() * 100 * 1.20)

# 图4-6b：作者上传视频数洛伦茨曲线
author_supply_values = np.sort(a["uploaded_video_count"].to_numpy())
author_supply_cum = (np.insert(np.cumsum(author_supply_values),0,0) / np.sum(author_supply_values))
axes[0, 1].plot(np.linspace(0, 1, len(author_supply_cum)),author_supply_cum,label=f"上传视频数（Gini={gini_author_video_supply:.3f}）")
axes[0, 1].plot([0, 1],[0, 1],linestyle="--",label="完全均匀供给")
axes[0, 1].set(title="作者上传视频数洛伦茨曲线",xlabel="作者累计占比",ylabel="上传视频累计占比",xlim=(0, 1),ylim=(0, 1))
axes[0, 1].legend()

# 图4-6c：头部作者曝光份额
sns.barplot(data=top_author_exposure,x="分组",y=top_author_exposure["曝光份额"] * 100,ax=axes[1, 0],edgecolor="black")
for i, row in top_author_exposure.iterrows():
    axes[1, 0].text(i,row["曝光份额"] * 100,f'{row["曝光份额"] * 100:.1f}%\n作者数：{int(row["作者数"])}', ha="center", va="bottom", fontsize=9)
axes[1, 0].set(title="头部作者曝光份额",xlabel="",ylabel="曝光份额（%）")
axes[1, 0].set_ylim(0, top_author_exposure["曝光份额"].max() * 100 * 1.20)

# 图4-6d：作者曝光次数洛伦茨曲线
author_exposure_values = np.sort(a["exposures"].to_numpy())
author_exposure_cum = (np.insert(np.cumsum(author_exposure_values),0,0) / np.sum(author_exposure_values))
axes[1, 1].plot(np.linspace(0, 1, len(author_exposure_cum)),author_exposure_cum,label=f"曝光次数（Gini={gini_author_exposure:.3f}）")
axes[1, 1].plot([0, 1],[0, 1],linestyle="--",label="完全均匀曝光")
axes[1, 1].set(title="作者曝光次数洛伦茨曲线",xlabel="作者累计占比",ylabel="曝光次数累计占比",xlim=(0, 1),ylim=(0, 1))
axes[1, 1].legend()

fig.suptitle("图4-6 作者作品供给与历史曝光集中度",fontsize=18,y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_06_author_supply_concentration.png",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 作者上传视频数较为均衡：作者上传视频数Gini仅为0.127，与前面分析得到大部分作者只上传1个视频的结论相符；
* 作者曝光次数明显向头部集中：作者作品曝光Gini达到0.723，Top 20%作者获得75.7%的曝光，说明本数据集历史曝光明显集中在少数作者的作品上。

### 4.3.3 作者历史观看与互动表现(基于曝光次数门槛)
对于作品有足够曝光次数的作者，分析其作品观看和互动表现

In [ ]:
# （1）设置作者最低曝光次数门槛
AUTHOR_EXP_THRESHOLD = int(a.loc[a["exposures"] > 0,"exposures"].quantile(0.30))    # 作者作品总曝光次数P30门槛
a_exp = a[a["exposures"] >= AUTHOR_EXP_THRESHOLD].copy()                            # 保留达到最低曝光门槛的作者

print(f"最低曝光门槛={AUTHOR_EXP_THRESHOLD}")
print(f"保留作者={len(a_exp)}，占全部作者{len(a_exp)/len(a)*100:.1f}%")
print(f"占有曝光作者{len(a_exp)/(a['exposures']>0).sum()*100:.1f}%")

# （2）作者作品观看与互动行为表现描述统计
display(a_exp[view_cols].describe().style.format("{:.2f}"))
display(a_exp[view_rate_cols].describe().style.format("{:.4f}"))
display(a_exp[interaction_cols].describe().style.format("{:.4f}"))
display(a_exp[interaction_rate_cols].describe().style.format("{:.6f}"))

In [ ]:
# （3）作者作品整体每万次曝光互动行为次数
author_interaction_per_10k = pd.DataFrame({
    "行为": ["点赞","评论","转发","点踩","进入主页","关注"],
    "每万次曝光行为次数": [
        a_exp["likes"].sum()          / a_exp["exposures"].sum()*10000,
        a_exp["comments"].sum()       / a_exp["exposures"].sum()*10000,
        a_exp["forwards"].sum()       / a_exp["exposures"].sum()*10000,
        a_exp["hates"].sum()          / a_exp["exposures"].sum()*10000,
        a_exp["profile_enters"].sum() / a_exp["exposures"].sum()*10000,
        a_exp["follows"].sum()        / a_exp["exposures"].sum()*10000
    ]
})

In [ ]:
## 图4-7：作者作品历史观看与互动表现
fig = plt.figure(figsize=(14,10))
gs = fig.add_gridspec(2,2)
ax1 = fig.add_subplot(gs[0,0])
ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[1,:])

## 图4-7a：作者作品观看行为率
author_view_plot = a_exp[["valid_view_rate","long_view_rate","complete_view_rate"]].rename(columns={"valid_view_rate":"有效观看率","long_view_rate":"长播放率","complete_view_rate":"完播率"})
sns.boxplot(data=author_view_plot,showfliers=False,ax=ax1)
ax1.set(title="作品观看行为率",xlabel="",ylabel="行为率",ylim=(0,1.05))

## 图4-7b：作品每次曝光平均播放时长
author_avg_play_log, author_avg_play_ticks, author_avg_play_labels = log1p_axis_values(a_exp["avg_play_time_per_exposure_s"])
sns.histplot(x=author_avg_play_log,bins=20,ax=ax2,edgecolor="black")
ax2.axvline(np.log1p(a_exp["avg_play_time_per_exposure_s"].median()),color="red",linestyle="--",label=f'中位数={a_exp["avg_play_time_per_exposure_s"].median():.1f}')
ax2.set_xticks(author_avg_play_ticks)
ax2.set_xticklabels(author_avg_play_labels)
ax2.set(title="作品每次曝光平均播放时长",xlabel="秒 / 次曝光",ylabel="作者数")
ax2.legend()

## 图4-7c：作品互动行为整体表现
sns.barplot(data=author_interaction_per_10k,x="行为",y="每万次曝光行为次数",ax=ax3,edgecolor="black")
ax3.bar_label(ax3.containers[0],fmt="%.1f",padding=4)
ax3.set(title="作品互动行为整体表现",xlabel="",ylabel="每万次曝光行为次数（次）")
ax3.set_ylim(0,author_interaction_per_10k["每万次曝光行为次数"].max()*1.15)

fig.suptitle(f"图4-7 作者作品历史观看与互动表现（曝光次数≥{AUTHOR_EXP_THRESHOLD}，作者数量={len(a_exp)}）",fontsize=18,y=1.02)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_07_author_view_interaction.png",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 达到最低曝光门槛的作者中，作品有效观看率、长播放率和完播率中位数分别约40.0%、28.0%和8.6%，与视频侧整体表现接近；
* 作者作品平均每次曝光的播放时长中位数为17.3s，与视频级表现（17s）也基本一致；
* 用户互动行为同样主要集中在主页进入和点赞，其余互动行为较为稀疏；

作者级与视频级观看、互动结果整体较为接近，符合预期：
* 一方面，两者来源于同一前期历史窗口交互日志；另一方面，样本中约87%的作者仅上传1个视频，因此作者级聚合在较大程度上保留了视频级表现特征。
* 两者仍分别保留，是因为视频级指标用于描述单条视频的历史表现，作者级指标则用于评价作者全部作品的整体历史表现，并服务于后续作者分层。

### 4.3.4 作者历史观看与互动表现分层
基于历史窗口中作者作品的观看表现与互动表现构建二维作者分层。
1. 首先以作者作品累计历史曝光次数的P30作为最低观测门槛，筛选正式分层作者，其余作者保留为低曝光次数或零曝光类型；
2. 分别根据作者历史的观看行为指标与互动行为指标构建view_score与interaction_score，用于区分作者作品在观看和互动反馈两方面的表现；
3. 采用view_score与interaction_score各自P70作为高水平切点，将正式分层作者划分为四种类型。

In [ ]:
# （1）设置正式分层作者样本
a_eligible = a_exp.copy()        # 复制达到曝光门槛的作者数据，进行分层

# （2）为观看与互动表现赋分
a_eligible['avg_play_time_score'] = rank_score(a_eligible["avg_play_time_per_exposure_s"])  #rank_score函数已在3.2.4用户分层中定义
a_eligible["valid_view_score"]    = rank_score(a_eligible["valid_view_rate"])
a_eligible["long_view_score"]     = rank_score(a_eligible["long_view_rate"])
a_eligible["complete_view_score"] = rank_score(a_eligible["complete_view_rate"])
a_eligible["like_score"]          = rank_score(a_eligible["like_rate"])
a_eligible["forward_score"]       = rank_score(a_eligible["forward_rate"])
a_eligible["profile_enter_score"] = rank_score(a_eligible["profile_enter_rate"])
a_eligible["follow_score"]        = rank_score(a_eligible["follow_rate"])
a_eligible["hate_score"]          = rank_score(a_eligible["hate_rate"])

# （3）观看、互动表现得分（赋分规则同4.2.4）
a_eligible["view_score"] = (
                              a_eligible["avg_play_time_score"] * 0.15  #次均播放时长得分
                            + a_eligible["valid_view_score"]    * 0.20  #有效观看得分
                            + a_eligible["long_view_score"]     * 0.30  #长播放得分
                            + a_eligible["complete_view_score"] * 0.35  #完整播放得分
                            )
a_eligible["interaction_score"] = (
                              a_eligible["like_score"]          * 0.15   #点赞行为得分
                            + a_eligible["forward_score"]       * 0.10   #转发行为得分
                            + a_eligible["profile_enter_score"] * 0.30   #进入主页行为得分
                            + a_eligible["follow_score"]        * 0.45   #关注行为得分
                            - a_eligible["hate_score"]          * 0.30   #点踩行为得分（负值）
                            )
a_eligible["interaction_score"] = rank_score(a_eligible["interaction_score"])

# （4）划分观看表现、互动表现得分分层阈值：采用正式分层作者观看表现得分P70和互动表现得分P70作为分层切点
AUTHOR_VIEW_CUTOFF = a_eligible["view_score"].quantile(0.70)
AUTHOR_INTERACTION_CUTOFF = a_eligible["interaction_score"].quantile(0.70)

print(f"作者观看表现P70分层切点={AUTHOR_VIEW_CUTOFF:.4f}")
print(f"作者互动表现P70分层切点={AUTHOR_INTERACTION_CUTOFF:.4f}")


m_hi_view = a_eligible["view_score"] >= AUTHOR_VIEW_CUTOFF
m_hi_interaction = a_eligible["interaction_score"] >= AUTHOR_INTERACTION_CUTOFF

# (5)进行作者分层
a_eligible["author_segment"] = "一般历史表现作者"
a_eligible.loc[m_hi_view & m_hi_interaction,"author_segment"] = "综合高表现作者"
a_eligible.loc[m_hi_view & ~m_hi_interaction,"author_segment"] = "观看优势型作者"
a_eligible.loc[~m_hi_view & m_hi_interaction,"author_segment"] = "互动优势型作者"

# （6）将作者分层标签与得分写回全量作者
author_seg = a.copy()                           # 取全量作者进行类型划分
author_seg["author_segment"] = "无历史曝光作者"    # 设置默认作者类型
author_seg.loc[(author_seg["exposures"] > 0) & (author_seg["exposures"] < AUTHOR_EXP_THRESHOLD),"author_segment"] = "低曝光次数作者"

author_score_cols = ["avg_play_time_score",
                     "valid_view_score",
                     "long_view_score",
                     "complete_view_score",
                     "like_score",
                     'forward_score',
                     "profile_enter_score",
                     "follow_score",
                     "hate_score",
                     "view_score",
                     "interaction_score"]

author_seg[author_score_cols] = a_eligible[author_score_cols]
author_seg.loc[a_eligible.index, "author_segment"] = a_eligible["author_segment"]

# （7）输出作者历史分层结果
author_seg_order = ["综合高表现作者","观看优势型作者","互动优势型作者","一般历史表现作者","低曝光次数作者","无历史曝光作者"]  # 固定作者类型顺序

author_segment_result = author_seg["author_segment"].value_counts().reindex(author_seg_order).rename("作者数").reset_index() # 各作者类型的视频数量
author_segment_result["占全部作者%"] = author_segment_result["作者数"] / len(author_seg) * 100 # 各视频类型的视频数占全部视频数比例
display(author_segment_result.round(2))

In [ ]:
## 图4-8：作者历史表现分层结果
fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(data=author_segment_result,x="作者数",y="author_segment",order=author_seg_order,ax=ax,edgecolor="black")
for i,row in author_segment_result.set_index("author_segment").reindex(author_seg_order).reset_index().iterrows():
    ax.text(row["作者数"],i,f'作者人数：{int(row["作者数"])} (占全体作者：{row["占全部作者%"]:.2f}%)',va="center",fontsize=9)
ax.set(title="图4-8 作者历史表现分层结果",xlabel="作者数",ylabel="")
ax.set_xlim(0,author_segment_result["作者数"].max()*1.20)
fig.tight_layout()
fig.savefig(OUT_PHASE4 / "P4_08_author_segmentation.png",dpi=300,bbox_inches="tight")
plt.show()
plt.close()

本阶段不再额外设置“不同作者类型画像”章节。作者分层已经使用观看和互动表现指标，相关字段不再用于分层后的再次解释。

# Phase 5. 标准推荐与随机曝光的视频曝光分配及用户反馈差异分析
KuaiRand官方指出，真实推荐日志会受到exposure bias影响：用户看到的视频已经经过推荐系统选择，因此日志中的反馈不能视为对候选视频的随机观察。

为获得相对更少受推荐选择影响的参照，官方在2022-04-22～2022-05-08的正常推荐流中，以固定概率将原推荐位置替换为从7583个候选视频中均匀随机抽取的视频，并记录用户反馈。

因此，本阶段不比较standard与random谁“更好”，而是利用random exposure作为参照，分析：

1. standard相对随机曝光选择了什么视频；
2. 两种曝光来源下观察到的观看与互动行为反馈有什么差异；
3. 固定同一用户或同一视频后，这些差异是否仍存在；
4. 整体结果是否掩盖了不同历史用户 / 视频类型的差异。

本阶段结果用于描述standard推荐日志中的选择性曝光特征。

本节采用α = 0.05作为显著性水平。

## 5.1 分析准备

### 5.1.1 准备历史用户、视频和作者标签
把Phase 3、4得到的用户、视频、作者历史标签写入MySQL。

In [ ]:
# （1）将Phase 3用户历史类型写入MySQL
user_seg[["user_id","user_segment"]].to_sql("ads_user_segment_pre", con=engine, if_exists="replace", index=False)

# （2）将Phase 4视频历史类型写入MySQL
video_seg[["video_id","author_id","video_segment"]].to_sql("ads_video_segment_pre", con=engine, if_exists="replace", index=False)

# （3）将Phase 4作者历史类型写入MySQL
author_seg[["author_id","author_segment"]].to_sql("ads_author_segment_pre", con=engine, if_exists="replace", index=False)

### 5.1.2 SQL：构建用户/视频/作者后期表现表
后续分析依赖的3张后期表现表在SQL文件中统一构建（见 Phase5_ADS_standard_random_analysis.sql）：
1. ads_user_post_behavior_feature（用户后期行为表现表）：用于分析不同曝光来源下用户观看及互动反馈差异
2. ads_video_post_behavior_feature（视频后期行为表现表）：用于分析不同曝光来源下视频获得用户反馈的差异。
3. ads_author_segment_exposure_share（作者作品后期曝光分配表）：由于作者层面的观看及互动反馈本质上由其作品表现汇总得到，与视频层面的表现分析存在较强关联，因此仅分析不同历史作者类型在两种曝光来源下的曝光分配差异。

In [ ]:
# （1）读取后期表现表
users_post = pd.read_sql('SELECT * FROM ads_user_post_behavior_feature', con=engine)
videos_post = pd.read_sql('SELECT * FROM ads_video_post_behavior_feature', con=engine)
authors_post = pd.read_sql('SELECT * FROM ads_author_segment_exposure_share', con=engine)

# （2）检查一下这3个表
display(users_post.describe().style.format('{:.2f}'))
display(videos_post.describe().style.format('{:.2f}'))
display(authors_post.describe().style.format('{:.2f}'))

### 5.1.3 确定反馈指标与配对检验方法
为了后续对具体的指标在两种曝光来源下的差异进行比较，先进行以下准备工作：
1. 确定具体对比指标

In [ ]:
comparison_metrics = [   # 用户、视频后期行为表现对比指标
    ('valid_view_rate',    '有效观看率'),
    ('long_view_rate',     '长播放率'),
    ('complete_view_rate', '完播率'),
    ('avg_play_time_per_exposure_s', '每次曝光平均播放时长（秒）'),
    ('like_rate',          '点赞率'),
    ('comment_rate',       '评论率'),
    ('forward_rate',       '转发率'),
    ('profile_enter_rate', '主页进入率'),
    ('follow_rate',        '关注率'),
    ('hate_rate',          '点踩率')]
#    '指标名称',             '中文名'

2. 构造对比统计函数

In [ ]:
# (1)构造指标配对函数：取出同一个用户在两种曝光来源下，同一个指标的两个值
### 输出粒度：user_id * 标准推荐下指标值 * 随机曝光下指标值
def get_pair(data, id_cols, metric, ids = None):
    # data      -后期行为表
    # id_cols   -需要进行对比的id字段,用户为base_ids;视频为video_id;作者为author_id
    # metric    -需要对比的指标（字段）
    # ids       -只对比id.isin(ids); 默认无需输入，因为只有在进行用户级对比的时候需要筛选出 双曝光有效用户的id

    if ids is not None:
        data = data[data[id_cols].isin(ids)]    # 只有在ids内的id才进入分析

    std = data[data['log_source'] == 'standard_post'][[id_cols, metric]].rename(columns={metric: 'standard_post'})   # 取出  id * 标准推荐下指标值
    rnd = data[data['log_source'] == 'random_post'][[id_cols, metric]].rename(columns={metric: 'random_post'})       # 取出  id * 随机曝光下指标值
    wide = std.merge(rnd, on=id_cols, how='inner')    # 将std和rnd按照同一个id匹配，形成最终输出粒度
    return wide      # 返回配对好的表

In [ ]:
# （2）构造配对秩二列效应量函数：指标在哪种曝光来源下，数值更高，以及差异的大小
# 输出：范围在（-1,1）的效应量，正值代表standard的指标数值整体偏高，负值代表random偏高，绝对值越大说明差异越明显
def rank_biserial(diff):
    # 输入：同一个用户，某指标的 标准推荐值 - 随机曝光值
    ranks = rankdata(np.abs(diff), method='average')    # 将差值取绝对值并进行从小到大排序（输出秩）
    pos_sum = ranks[diff > 0].sum()                     # standard_post 数值更高的行的秩和（正秩和）
    neg_sum = ranks[diff < 0].sum()                     # random_post   数值更高的行的秩和（负秩和）
    denom = pos_sum + neg_sum                           # 总秩和
    if denom == 0:                                      # 如果分母为0则输出0
        return 0.0
    return(pos_sum - neg_sum) / denom  # 正负秩和差：正值代表standard的指标数值整体偏高，负值代表random偏高，绝对值越大说明差异越明显

## 5.2 用户侧反馈与视频曝光分配分析

1. 设置用户双侧曝光次数门槛
2. 分别运用rank_biserial和Wilcoxon检验用户各指标的差异
* Wilcoxon：检验两种曝光来源下，同一用户的指标差异是否具有统计显著性
* rank_biserial：衡量指标的差异整体偏向哪一边，以及差异的效应强弱

### 5.2.1 确定用户反馈比较的双曝光有效样本

In [ ]:
# （1）识别同时受到两种曝光来源的用户
n_user_sources   = users_post.groupby("user_id")["log_source"].nunique()        # 计算每个用户受到的曝光来源数
dual_user_ids    = n_user_sources[n_user_sources == 2].index                    # 受到两种双曝光来源的用户的id
n_standard_users = len(users_post[users_post['log_source']=='standard_post'])   # 标准推荐用户数
n_random_users   = len(users_post[users_post['log_source']=='random_post'])     # 随机曝光用户数

print(f"标准推荐用户: {n_standard_users}")
print(f"随机曝光用户: {n_random_users}")
print(f"双曝光来源用户: {len(dual_user_ids)},占标准推荐用户{len(dual_user_ids)/n_standard_users*100:.2f}%,占随机曝光用户{len(dual_user_ids)/n_random_users*100:.2f}%")


# （2）对于每个双曝光用户，取两侧曝光来源曝光次数的最小值，设置P30门槛
user_min_exp = users_post[users_post["user_id"].isin(dual_user_ids)].groupby("user_id")["exposures"].min()    # 用户双侧曝光来源曝光次数最小值
USER_PAIR_EXPOSURE_THRESHOLD = int(user_min_exp.quantile(0.30))                                     # 双曝光次数p30门槛

print(f"双侧曝光次数门槛 = {USER_PAIR_EXPOSURE_THRESHOLD}")


# （3）取出达到双曝光门槛的用户
base_user_id   = user_min_exp[user_min_exp >= USER_PAIR_EXPOSURE_THRESHOLD].index    # 双曝光有效用户的id

print(f"Phase 5 双曝光有效用户数: {len(base_user_id)},占标准推荐用户{len(base_user_id)/n_standard_users*100:.2f}%,占随机曝光用户{len(base_user_id)/n_random_users*100:.2f}%")

base_users = users_post[users_post['user_id'].isin(base_user_id)]        # 取出双曝光有效用户的数据

### 5.2.2 用户整体行为反馈差异（基于双曝光门槛）

In [ ]:
users_paired_comparison = []    # 空表格记录结果

for metric, label in comparison_metrics:
    # metric：指标名
    # label：指标中文名

# （1）取出对比指标
    wide = get_pair(base_users, 'user_id', metric, base_user_id)   # 输入后期用户表, 用户id（字段）, 双曝光有效用户id, 行为对比指标
    std = wide['standard_post']    # 标准推荐值
    rnd = wide['random_post']      # 随机曝光值

# （2）排除两种曝光来源下，指标都为0的组：如果两边都没发生这个行为，没有对比的意义
    if std.sum() == 0 and rnd.sum() == 0:
        continue

# （3）调用rank_biserial统计函数，计算效应量
    rb = rank_biserial(std - rnd)
    # 输出一个（-1,1）之间的值

# （4）调用wilcoxon统计函数，计算指标差异显著性
    stat, p, *_ = wilcoxon(std, rnd, zero_method='pratt', alternative='two-sided')
    # stat:Wilcoxon 检验统计量
    # p:p值
    # *_: 忽略其他返回的参数

# （5）将一个指标的计算结果存入表格中
    users_paired_comparison.append({
        '指标': label,
        '配对用户数': len(std),
        '标准推荐均值': std.mean(),
        '随机曝光均值': rnd.mean(),
        '平均值配对差': (std - rnd).mean(),
        'p_value': p,
        'rank_biserial': rb,
    })

# （6）调用multipletests函数，将指标的p值进行BH-FDR校正
users_paired_comparison = pd.DataFrame(users_paired_comparison)
users_paired_comparison['q_fdr'] = multipletests(users_paired_comparison['p_value'], method='fdr_bh')[1]
users_paired_comparison['FDR显著'] = users_paired_comparison['q_fdr'] < 0.05    # 以95%水平判断是否显著

# 展示结果
display(users_paired_comparison[['指标','配对用户数','标准推荐均值','随机曝光均值','平均值配对差','rank_biserial','p_value','q_fdr','FDR显著']].round(6))

users_paired_comparison.to_csv(
    OUT_PHASE5/'P5_01_users_paired_comparison_result.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:
## 图5-1：用户整体指标在standard与random之间的配对效应量
users_paired_comparison.loc[users_paired_comparison["FDR显著"], "指标"] += " *"   # 显著指标名后加" *"

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=users_paired_comparison, x="rank_biserial", y="指标", ax=ax, color="royalblue")
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set(title="图5-1 用户整体行为反馈差异效应量（正值：standard整体更高，负值：random整体更高，*表示FDR显著）", xlabel="rank_biserial", ylabel="")
ax.set_xlim(-1, 1)

fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_01_user_overall_effect.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 对于同一批双曝光有效用户：除转发率外，standard下各观看、互动行为指标明显更优；
* 观看指标的效应量明显大于互动指标；

### 5.2.3 不同历史用户类型行为反馈差异（基于双曝光门槛）
对于不同的历史用户类型，其在两种曝光来源下表现出的行为差异

In [ ]:
# 逐个历史用户类型，重复用户级配对检验
users_seg_paired_comparison = []

for seg in user_seg_order:  # 取出各个历史用户类型
    user_seg_id = base_users.loc[base_users['user_segment'] == seg, 'user_id']   # 双曝光有效用户中，当前类型的用户ID

    for metric, label in comparison_metrics:
        wide = get_pair(base_users, 'user_id', metric, user_seg_id)
        std = wide['standard_post']
        rnd = wide['random_post']

        if std.sum() == 0 and rnd.sum() == 0:
            continue

        rb = rank_biserial(std - rnd)
        stat, p, *_ = wilcoxon(std, rnd, zero_method='pratt', alternative='two-sided')

        users_seg_paired_comparison.append({
            'user_segment': seg,
            '指标': label,
            '配对用户数': len(std),
            '标准推荐均值': std.mean(),
            '随机曝光均值': rnd.mean(),
            '平均值配对差': (std - rnd).mean(),
            'p_value': p,
            'rank_biserial': rb,
        })

users_seg_paired_comparison = pd.DataFrame(users_seg_paired_comparison)

users_seg_paired_comparison['q_fdr'] = multipletests(users_seg_paired_comparison['p_value'], method='fdr_bh')[1]
users_seg_paired_comparison['FDR显著'] = users_seg_paired_comparison['q_fdr'] < 0.05

users_seg_paired_comparison.to_csv(
    OUT_PHASE5/'P5_02_users_seg_paired_comparison_result.csv',
    index=False,
    encoding='utf-8-sig'
)

display(users_seg_paired_comparison[['user_segment','指标','配对用户数','标准推荐均值','随机曝光均值','平均值配对差','rank_biserial','p_value','q_fdr','FDR显著']].round(6))

In [ ]:
## 图5-2：不同历史用户类型下各指标的效应量热力图
cols = [label for _, label in comparison_metrics]
data = pd.DataFrame(index=user_seg_order, columns=cols, dtype=float)     # 数值
annot = pd.DataFrame(index=user_seg_order, columns=cols, dtype=object)   # 显示的文字

for _, row in users_seg_paired_comparison.iterrows():
    data.loc[row["user_segment"], row["指标"]] = row["rank_biserial"]
    annot.loc[row["user_segment"], row["指标"]] = f"{row['rank_biserial']:.2f}" + ("*" if row["FDR显著"] else "")

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(data, annot=annot, fmt="", cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0, ax=ax)
ax.set(title="图5-2 不同历史用户类型的行为反馈差异效应量（正值：standard整体更高，负值：random整体更高，*表示FDR显著）", xlabel="", ylabel="")
ax.tick_params(axis="x", rotation=35)
ax.tick_params(axis="y", rotation=0)

fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_02_user_segment_effect_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 对于六类历史用户类型，除转发率外所有指标都在standard下表现更优，说明用户整体层面的反馈差异不是由某一个历史用户类型单独造成；
* 各用户类型中，观看指标的效应量也普遍大于互动指标；

### 5.2.4 不同用户类型接收的视频类型曝光分布
从用户视角描述standard与random的视频曝光分布：（仅对双曝光来源用户分析）
对于同一历史用户类型，比较两种曝光来源下各历史视频类型的曝光份额。

In [ ]:
# （1）读取用户类型 × 视频类型曝光分布
user_video_share = pd.read_sql("SELECT * FROM ads_user_video_segment_exposure_distribution",con=engine)

# （2）将原表转宽表：每行为 user_segment × video_segment × standard下曝光份额 × random下曝光份额
user_video_share = user_video_share.pivot_table(
    index=["user_segment", "video_segment"],
    columns="log_source",
    values="exposure_share",
    fill_value=0
)
user_video_share["share_diff"] = (user_video_share["standard_post"] - user_video_share["random_post"]) * 100    # 计算曝光份额差

display(user_video_share.round(2))

In [ ]:
## 图5-3：不同用户类型观看的视频类型曝光份额差
data = user_video_share["share_diff"].unstack().reindex(index=user_seg_order, columns=video_seg_order)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(data, annot=True, fmt=".1f", cmap="coolwarm", center=0, ax=ax)
ax.set(title="图5-3 用户类型 × 视频类型百分比曝光份额差（standard - random）", xlabel="", ylabel="")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_03_user_video_share_diff.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 六类用户均呈现相似的视频曝光分布：standard中历史高曝光视频的份额更高，而random中一般曝光、低曝光和无历史曝光视频的份额更高；
* 说明相较于random，standard的曝光更集中于历史阶段已经获得较多standard曝光的视频。（这与standard由正常推荐系统进行视频选择、而random从候选池随机抽取视频的机制相一致）

### 5.2.5 用户侧综合分析结论
* 对于同一批用户，standard下的观看行为及大部分互动行为反馈明显更优，而且这一结果在六类历史用户中都存在；
* 但5.2.4分析结果表明，同一批用户在后期两种曝光来源下实际接收到的视频类型结构明显不同：standard更多集中于standard历史高曝光视频，而random更多覆盖一般曝光和低曝光视频。

因此，用户侧能够确认“行为反馈差异”和“视频曝光分配差异”同时存在，但仅固定用户比较仍无法判断差异究竟来自曝光来源本身、视频选择差异、用户—视频匹配以及其他未观测因素。

## 5.3 视频侧用户反馈与曝光分配分析

### 5.3.1 确定视频反馈比较的双曝光有效样本

In [ ]:
# （1）识别同时出现在两种曝光来源中的视频
n_video_sources     = videos_post.groupby("video_id")["log_source"].nunique()       # 后期每个视频的曝光来源数
dual_video_ids      = n_video_sources[n_video_sources == 2].index                   # 后期双曝光来源视频
n_standard_videos = len(videos_post[videos_post['log_source']=='standard_post'])    # 标准推荐来源视频数
n_random_videos   = len(videos_post[videos_post['log_source']=='random_post'])      # 随机曝光来源视频数

print(f"标准推荐视频: {len(videos_post[videos_post['log_source']=='standard_post'])}")
print(f"随机曝光视频: {n_random_videos}")
print(f"双曝光来源视频: {len(dual_video_ids)},占标准推荐视频{len(dual_video_ids)/n_standard_videos*100:.2f}%,占随机曝光视频{len(dual_video_ids)/n_random_videos*100:.2f}%")

# （2）对于每个双曝光视频，取两侧曝光来源曝光次数的最小值，设置P30门槛
video_min_exp = videos_post[videos_post["video_id"].isin(dual_video_ids)].groupby("video_id")["exposures"].min()    # 视频双侧曝光来源曝光次数最小值
VIDEO_PAIR_EXPOSURE_THRESHOLD = int(video_min_exp.quantile(0.30))                                                   # 双曝光次数p30门槛

print(f"双侧曝光次数门槛 = {VIDEO_PAIR_EXPOSURE_THRESHOLD}")

# （3）得到符合双曝光门槛的视频
base_video_ids = video_min_exp[video_min_exp >= VIDEO_PAIR_EXPOSURE_THRESHOLD].index   # 双曝光有效视频的id

print(f"Phase 5 双曝光有效视频数: {len(base_video_ids) },占标准推荐视频{len(base_video_ids) /n_standard_videos*100:.2f}%,占随机曝光视频{len(base_video_ids) /n_random_videos*100:.2f}%")

base_videos = videos_post[videos_post['video_id'].isin(base_video_ids)]                # 取出双曝光有效视频的数据

### 5.3.2 视频整体获得的用户反馈差异（基于双曝光门槛）

In [ ]:
videos_paired_comparison = []

for metric, label in comparison_metrics:

    wide = get_pair(base_videos,'video_id', metric)   # 输入后期视频表，视频id，行为对比指标
    std = wide['standard_post']    # 标准推荐值
    rnd = wide['random_post']      # 随机曝光值

    if std.sum() == 0 and rnd.sum() == 0:
        continue

    rb = rank_biserial(std - rnd)
    stat, p, *_ = wilcoxon(std, rnd, zero_method='pratt', alternative='two-sided')

    videos_paired_comparison.append({
        '指标': label,
        '配对视频数': len(std),
        '标准推荐均值': std.mean(),
        '随机曝光均值': rnd.mean(),
        '平均值配对差': (std - rnd).mean(),
        'p_value': p,
        'rank_biserial': rb,
    })

videos_paired_comparison = pd.DataFrame(videos_paired_comparison)
videos_paired_comparison['q_fdr'] = multipletests(videos_paired_comparison['p_value'], method='fdr_bh')[1]
videos_paired_comparison['FDR显著'] = videos_paired_comparison['q_fdr'] < 0.05    # 以95%水平判断是否显著

# 展示结果
display(videos_paired_comparison[['指标','配对视频数','标准推荐均值','随机曝光均值','平均值配对差','rank_biserial','p_value','q_fdr','FDR显著']].round(6))

videos_paired_comparison.to_csv(
    OUT_PHASE5/'P5_03_videos_paired_comparison_result.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:
## 图5-4：视频整体获得的用户反馈差异效应量
videos_paired_comparison.loc[videos_paired_comparison["FDR显著"], "指标"] += " *"

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=videos_paired_comparison, x="rank_biserial", y="指标", ax=ax, color="royalblue")
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set(title="图5-4 视频整体用户反馈差异效应量（正值：standard整体更高，负值：random整体更高，*表示FDR显著）", xlabel="rank_biserial", ylabel="")
ax.set_xlim(-1, 1)

fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_04_video_overall_effect.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 对于同一批双曝光有效视频：除转发率外，standard下所有观看、互动行为指标都表现更优，说明两种曝光来源展示的视频构成不同，不能完全解释用户侧观察到的观看反馈差异；
* 除点踩率外，观看行为指标效应量均明显大于互动行为效应量，这一结果与用户侧较为相似。

### 5.3.3 不同历史视频类型获得的用户反馈差异（基于双曝光门槛）

In [ ]:
# 逐个历史视频类型，重复视频级配对检验
videos_seg_paired_comparison = []

for seg in video_seg_order:                                            # 取出各个历史视频类型
    video_seg_id = base_videos.loc[base_videos['video_segment'] == seg, 'video_id']    # 取出该历史视频类型的所有视频id

    for metric, label in comparison_metrics:     # 取出对比字段
        wide = get_pair(base_videos, 'video_id', metric, video_seg_id)
        std = wide['standard_post']
        rnd = wide['random_post']

        if std.sum() == 0 and rnd.sum() == 0:
            continue

        rb = rank_biserial(std - rnd)
        stat, p, *_ = wilcoxon(std, rnd, zero_method='pratt', alternative='two-sided')

        videos_seg_paired_comparison.append({
            'video_segment': seg,
            '指标': label,
            '配对视频数': len(std),
            '标准推荐均值': std.mean(),
            '随机曝光均值': rnd.mean(),
            '平均值配对差': (std - rnd).mean(),
            'p_value': p,
            'rank_biserial': rb,
        })

videos_seg_paired_comparison = pd.DataFrame(videos_seg_paired_comparison)


videos_seg_paired_comparison['q_fdr'] = multipletests(videos_seg_paired_comparison['p_value'], method='fdr_bh')[1]
videos_seg_paired_comparison['FDR显著'] = videos_seg_paired_comparison['q_fdr'] < 0.05

videos_seg_paired_comparison.to_csv(
    OUT_PHASE5/'P5_04_videos_seg_paired_comparison_result.csv',
    index=False,
    encoding='utf-8-sig'
)

display(videos_seg_paired_comparison[['video_segment','指标','配对视频数','标准推荐均值','随机曝光均值','平均值配对差','rank_biserial','p_value','q_fdr','FDR显著']].round(6))

In [ ]:
## 图5-5：不同历史视频类型的用户反馈差异效应量
cols = [label for _, label in comparison_metrics]
data = pd.DataFrame(index=video_seg_order[:-1], columns=cols, dtype=float)    # 由于"无历史曝光视频"的类型在后期只存在1个得到曝光的视频，顾不对该类型进行对比分析
annot = pd.DataFrame(index=video_seg_order[:-1], columns=cols, dtype=object)

for _, row in videos_seg_paired_comparison[videos_seg_paired_comparison["video_segment"] != "无历史曝光视频"].iterrows():
    data.loc[row["video_segment"], row["指标"]] = row["rank_biserial"]
    annot.loc[row["video_segment"], row["指标"]] = f"{row['rank_biserial']:.2f}" + ("*" if row["FDR显著"] else "")

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(data, annot=annot, fmt="", cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0, ax=ax)
ax.set(title="图5-5 不同历史视频类型的用户反馈差异效应量（正值：standard整体更高，负值：random整体更高，*表示FDR显著）", xlabel="", ylabel="")
ax.tick_params(axis="x", rotation=35)
ax.tick_params(axis="y", rotation=0)

fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_05_video_segment_effect_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* 除无历史曝光视频因有效样本仅1个而不作分析外，其余5类视频的所有观看行为指标和点踩率都表现为standard更优；
* 互动行为反馈明显依赖视频历史类型：历史高曝光或高表现视频中，多数互动行为指标在standard下更优，而一般曝光一般表现视频和低曝光次数视频中，多项互动行为指标反而在random下更优；

综上，对于观看行为指标，其反馈差异与视频整体反馈差异基本一致；而互动行为指标则明显受到视频历史类型影响。

### 5.3.4 不同历史视频类型在后期两种曝光来源下的曝光分配

In [ ]:
# （1）构建视频类型曝光份额表：
# 各历史视频类型在三种窗口下的曝光次数
video_segment_exposure = pd.read_sql(
    """
    SELECT s.video_segment, d.log_source, SUM(d.exposures) AS exposures
    FROM dws_video_window_metrics AS d
    LEFT JOIN ads_video_segment_pre AS s ON d.video_id = s.video_id
    WHERE d.log_source IN ('standard_pre', 'standard_post', 'random_post')
    GROUP BY s.video_segment, d.log_source
    """,
    con=engine
)

# 转宽表：video_segment × 三种窗口曝光次数
video_segment_exp_wide = (
    video_segment_exposure
    .pivot(index="video_segment", columns="log_source", values="exposures")
    .reindex(video_seg_order)
    .fillna(0)
)

# 每种窗口内部重算曝光份额
video_segment_share_wide = (
    video_segment_exp_wide
    .div(video_segment_exp_wide.sum(axis=0), axis=1)
    [["standard_pre", "standard_post", "random_post"]]
)

# 整理展示表
video_segment_exposure_result = video_segment_share_wide.rename(columns={
    "standard_pre": "历史曝光份额",
    "standard_post": "standard曝光份额",
    "random_post": "random曝光份额"
})

display(video_segment_exposure_result.style.format("{:.2%}"))

In [ ]:
## 图5-6：不同历史视频类型在前后期的曝光份额
video_segment_share_wide.loc[video_seg_order].mul(100).plot(kind="bar", figsize=(12, 6), rot=20)
ax = plt.gca()
ax.set(title="图5-6 不同历史视频类型在前后期的曝光份额", xlabel="", ylabel="曝光份额（%）")
ax.legend(["历史standard", "standard", "random"], title="窗口")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=2, fontsize=8)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_06_video_segment_exposure_share.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

* standard_post 与历史 standard_pre 的视频曝光结构较为接近，无历史曝光视频在 standard_post 中仍未获得曝光；
* random_post 下的分配结构则明显不同，两类历史高曝光视频的曝光份额均下降至约 10.0%，而一般曝光一般表现视频和低曝光次数视频的曝光份额分别上升至 37.8% 和 30.2%。说明 random 相比 standard 明显增加了对历史一般曝光和低曝光视频的覆盖。

### 5.3.5 视频侧综合分析结论
* 固定同一批视频后，standard下的主要观看行为反馈仍明显更优，而且这一结果在5类视频历史类型中都存在；
* 从整体曝光分配看，standard_post 与历史 standard_pre 较为接近，而 random_post 明显降低了历史高曝光视频的曝光份额，并增加了对一般曝光和低曝光视频的覆盖；
* 与观看指标不同，互动指标会随着视频历史类型发生明显变化：历史高曝光或高表现的视频中，多项互动指标在standard下更高；而一般曝光一般表现视频和低曝光次数视频中，多项互动指标反而在random下更高。

## 5.4 作者侧作品曝光分配分析
由于作者本身不直接参与曝光，作者表现差异主要反映其作品（视频）表现差异，因此作者侧只分析作者类型的流量分配情况，不单独比较作者表现。

### 5.4.1 不同历史作者类型的作品曝光份额

In [ ]:
# （1）读取作者类型曝光份额分配表
author_share = pd.read_sql("SELECT * FROM ads_author_segment_exposure_share", con=engine)

# 转宽表：
author_share = author_share.pivot_table(
    index="author_segment",
    columns="log_source",
    values="exposure_share",
    fill_value=0
)
author_share["share_diff"] = (author_share["standard_post"] - author_share["random_post"]) * 100

display(author_share.round(2))

In [ ]:
## 图5-7：不同历史作者类型在两种曝光来源下的曝光份额
fig, ax = plt.subplots(figsize=(11, 6))

author_share.loc[author_seg_order, ["standard_post", "random_post"]].mul(100).plot(kind="bar", ax=ax, rot=20)
ax.legend(["standard", "random"], title="曝光来源")
ax.set(title="图5-7 不同历史作者类型的曝光份额", xlabel="", ylabel="曝光份额（%）")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=2, fontsize=8)

fig.tight_layout()
fig.savefig(OUT_PHASE5 / "P5_07_author_share_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

分析结果：
* standard中综合高表现、观看优势型和互动优势型作者的曝光份额均高于random；
* random中低曝光次数作者、一般历史表现作者的作品曝光份额均高于standard，其中低曝光次数作者曝光份额差异最大。
* 作者侧的曝光分配方向与视频侧较为接近：standard 对历史曝光较充分或表现较好的作者作品分配了更高曝光份额，而 random 对历史低曝光作者作品的覆盖更多。由于约 87% 的作者仅上传 1 个视频，且作者与视频分层使用的核心历史指标相近，因此作者侧结果与视频侧存在较强对应关系。

## 5.5 主要结论

1. 后期standard与random表现出明显的视频曝光分配差异：
对同一批双来源用户，standard下用户接触到更多历史高曝光视频；random下用户则接触到更多历史一般曝光、低曝光视频。从整体视频曝光份额看，standard_post 与历史 standard_pre 的视频曝光结构较为接近，而在 random_post 中，历史高曝光视频的曝光份额下降明显，一般曝光一般表现视频和低曝光次数视频的曝光份额则上升明显（相较于standard_pre）。

2. 在结论1的视频曝光分配差异下，两种曝光来源下用户反馈也存在明显差异：
对于同一批用户，standard下的有效观看率、长播放率、完播率和每次曝光平均播放时长等观看行为指标都明显更高，而且在六类历史用户类型中呈现这一结果。除转发率外，点赞、评论等互动行为指标也表现为standard更优（六类用户类型中也都呈现这一结果），但互动行为指标的效应量整体弱于观看行为指标。

3. 对同一批视频进行比较发现，结论1中的视频曝光分配差异，不能完全解释结论2中的观看反馈差异：
 固定同一批视频进行比较发现，同一个视频在standard下的观看行为指标仍然整体更优，而且这一结果在5类视频类型中都存在（互动行为反馈则存在视频类型差异）。说明standard下用户观看反馈更优，并不只是因为存在不同视频类型的曝光份额差。

4. 不同历史视频类型的互动反馈结果并不一致：
对于历史高曝光或高表现视频，多项行为互动指标在standard下表现更优；而对于一般曝光一般表现视频和低曝光次数视频，多项行为互动指标在random下表现更优。

5. 上述结果不能直接说明 standard 本身造成了更好的反馈：
目前 standard 与 random 下的用户和视频组合并不完全一致，因此当前分析主要说明两种曝光来源下存在关联差异；如果要进一步验证曝光方式是否真正造成反馈差异，需要设计随机 A/B Test。

6.  综合来看，两种曝光方式形成了明显不同的用户—视频匹配结果：
standard 下用户更多接触到历史上已经获得较多曝光的视频和作者作品，random 则让更多历史曝光较少的视频获得展示机会。用户最终产生的观看和互动反馈，不仅和视频本身有关，也会受到曝光方式以及实际触达的用户构成影响。因此，只根据 standard 历史日志中的曝光量和反馈结果，不能直接把“曝光少、反馈低”理解为视频本身质量差。

# Phase 6. 标准推荐场景长播放预测
利用standard_pre中的用户历史行为、视频历史表现、用户—视频兴趣匹配及当前视频属性，预测standard_post中一次用户—视频曝光是否产生long_view。
* 模型样本中的输入特征均来自预测窗口之前的历史信息。对standard_post预测样本进行分层随机划分，其中训练集用于训练LightGBM，验证集用于选择分类阈值，测试集仅用于最终模型效果评价。

## 6.1 SQL：构建模型样本
用户与视频历史特征均来自standard_pre，作为预测时可获得的历史信息；standard_post中每次用户—视频曝光作为模型样本，long_view为二分类标签。对全部standard_post样本按60%/20%/20%分层随机划分训练集、验证集、测试集。

## 6.2 划分训练集、验证集、测试集

In [ ]:
# （1）读取Phase 6模型样本
model_sample = pd.read_sql('SELECT * FROM ads_long_view_sample', con=engine)

# （2）对整个样本随机划分60%训练集、20%验证集、20%测试集
# 先划分20%样本做测试集
train, test = train_test_split(
    model_sample,
    test_size=0.20,     # 测试集占比 20%
    random_state=77,    # 固定随机结果（便于复现）
    stratify=model_sample['target']   # 按照target的正负比例进行分层抽样
)
# 再从剩下的80%样本中划分25%作为验证集
train, valid = train_test_split(
    train,
    test_size=0.25,
    random_state=77,
    stratify=train['target']
)

# （3）检查样本规模和 long_view 率
print(f"训练集样本数：{len(train)}，long_view率：{train['target'].mean():.4f}")
print(f"验证集样本数：{len(valid)}，long_view率：{valid['target'].mean():.4f}")
print(f"测试集样本数：{len(test)}，long_view率：{test['target'].mean():.4f}")

## 6.3 LightGBM训练

In [ ]:
# （1）构建特征表（共14个特征）
features = [
    # 用户历史行为
    'user_hist_exposures',
    'user_hist_valid_view_rate',
    'user_hist_long_view_rate',
    'user_hist_complete_view_rate',
    'user_hist_avg_play_time_per_exposure_s',
    # 用户历史活跃特征
    'active_days',
    # 用户兴趣匹配
    'top1_tag_match',
    'top2_tag_match',
    # 视频历史表现
    'video_hist_exposures',
    'video_hist_valid_view_rate',
    'video_hist_long_view_rate',
    'video_hist_complete_view_rate',
    'video_hist_avg_play_time_per_exposure_s',
    # 当前视频属性
    'duration_ms',
]

# （2）设置目标变量
y_train = train['target']   # 训练集标签
y_valid = valid['target']   # 验证集标签
y_test  = test['target']    # 测试集标签

# （3）建立 LightGBM 二分类模型
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,   # 训练200棵树（基学习器）
    learning_rate=0.05, # 学习率：每棵新树对前面模型结果进行多大幅度的修正
    num_leaves=31,      # 每棵树做多31个叶节点（默认值）
    random_state=77,    # 固定随机过程（方便复现）
    n_jobs=-1,          # 用所有 CPU 并行训练
    verbose=-1          # 不打印训练日志
);

In [ ]:
# （4）训练模型：利用训练集对模型进行训练
lgb_model.fit(train[features],      # 训练集特征
              y_train)              # 训练集真实标签
print('训练完成')

## 6.4 验证集分类阈值选择
使用验证集比较不同分类阈值下的Precision、Recall与F1；由于当前没有明确的误判与漏判业务成本，因此采用F1最高作为阈值选择规则，在兼顾Precision和Recall的情况下确定最终分类阈值。

In [ ]:
# （1）使用已经训练好的LightGBM预测验证集发生long_view的概率
lgb_pred_valid = lgb_model.predict_proba(valid[features])[:, 1]

# （2）查看验证集ROC-AUC
valid_auc = roc_auc_score(y_valid, lgb_pred_valid)

print(f"验证集ROC-AUC：{valid_auc:.4f}")

In [ ]:
# （3）逐个阈值计算验证集Precision / Recall / F1
thresholds = np.arange(0.01, 1.00, 0.01)    # 设置候选分类阈值：0.01~0.99，每次增加0.01

threshold_results = []

for threshold in thresholds:
    y_pred_valid = (lgb_pred_valid >= threshold).astype(int)
    threshold_results.append({
        'threshold': threshold,                                                 # 分类阈值
        'precision': precision_score(y_valid, y_pred_valid, zero_division=0),   # 精确率
        'recall': recall_score(y_valid, y_pred_valid, zero_division=0),         # 召回率
        'f1': f1_score(y_valid, y_pred_valid, zero_division=0)                  # F1分数
    })

threshold_result = pd.DataFrame(threshold_results)

display(threshold_result.round(4))

In [ ]:
# （4）确定最终用于测试集的分类阈值
best_threshold_row = threshold_result.loc[threshold_result['f1'].idxmax()]  # 找到验证集中F1最高的一行

BEST_THRESHOLD = float(best_threshold_row['threshold'])

print(f"验证集选定阈值：{BEST_THRESHOLD:.2f}")
print(f"对应Precision：{best_threshold_row['precision']:.4f}")
print(f"对应Recall：{best_threshold_row['recall']:.4f}")
print(f"对应F1：{best_threshold_row['f1']:.4f}")

In [ ]:
## 图6-1：不同阈值下的Precision/Recall/F1
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(threshold_result['threshold'], threshold_result['precision'], marker='o', label='Precision')
ax.plot(threshold_result['threshold'], threshold_result['recall'],    marker='o', label='Recall')
ax.plot(threshold_result['threshold'], threshold_result['f1'],        marker='o', label='F1')
ax.axvline(BEST_THRESHOLD, color='red', linestyle='--', label=f'选定阈值={BEST_THRESHOLD:.2f}')
ax.set(xlabel='分类阈值', ylabel='分数', title='图6-1 不同阈值下Precision/Recall/F1（验证集）')
ax.legend()

fig.tight_layout()
fig.savefig(OUT_PHASE6/'P6_01_threshold_metrics.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

分析结果：
* LightGBM在验证集上的ROC-AUC为0.7339。
* 验证集F1在阈值0.30时最高，为0.5751；对应Precision为0.4687、Recall为0.7441，因此固定0.30作为最终分类阈值。

## 6.5 测试集最终评价
模型结构和分类阈值均已在测试集评价前确定，测试集不再参与任何模型或阈值训练，仅用于评价模型对未参与训练与选择样本的最终泛化表现。

### 6.5.1 baseline与LightGBM ROC-AUC

In [ ]:
# （1）使用训练集用户历史long_view_rate中位数填充测试集中的缺失值
long_view_median = train['user_hist_long_view_rate'].median()

# （2）直接使用用户历史long_view_rate作为baseline预测分数
baseline_score = test['user_hist_long_view_rate'].fillna(long_view_median)

# （3）计算baseline测试集ROC-AUC
baseline_auc = roc_auc_score(y_test, baseline_score)

# （4）LightGBM预测测试集long_view概率
lgb_pred_test = lgb_model.predict_proba(test[features])[:, 1]

# （5）计算LightGBM测试集ROC-AUC
lgb_auc = roc_auc_score(y_test, lgb_pred_test)

# （6）输出最终ROC-AUC比较
print(f"历史long_view_rate baseline测试集ROC-AUC：{baseline_auc:.4f}")
print(f"LightGBM测试集ROC-AUC：{lgb_auc:.4f}")
print(f"LightGBM相较baseline绝对提升：{lgb_auc-baseline_auc:.4f}")

### 6.5.2 选定阈值下的测试集最终分类表现

In [ ]:
# （1）使用验证集已经选定的阈值，将测试集预测概率转换为0/1结果
y_pred_test = (lgb_pred_test >= BEST_THRESHOLD).astype(int)

# （2）计算最终测试集分类指标
test_precision = precision_score(y_test, y_pred_test, zero_division=0)
test_recall = recall_score(y_test, y_pred_test, zero_division=0)
test_f1 = f1_score(y_test, y_pred_test, zero_division=0)

# （3）计算Lift：模型预测为long_view样本的实际long_view率，相对于测试集整体long_view率的提升倍数
test_base_rate = y_test.mean()
test_lift = test_precision / test_base_rate

# （4）整理最终测试结果
final_test_result = pd.DataFrame([{
    'selected_threshold': BEST_THRESHOLD,
    'test_positive_rate': test_base_rate,
    'precision': test_precision,
    'recall': test_recall,
    'f1': test_f1,
    'lift': test_lift
}])

display(final_test_result.round(4))

In [ ]:
# 图6-2：测试集Baseline与LightGBM ROC曲线

# 分别计算Baseline和LightGBM的ROC曲线
fpr_base, tpr_base, _ = roc_curve(y_test, baseline_score)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, lgb_pred_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_base, tpr_base, label=f'历史long_view_rate基线 (AUC={baseline_auc:.3f})')
ax.plot(fpr_lgb, tpr_lgb, label=f'LightGBM (AUC={lgb_auc:.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', label='随机猜测')
ax.set(xlabel='假正率', ylabel='真正率', title='图6-2 测试集ROC曲线')
ax.legend()

fig.tight_layout()
fig.savefig(OUT_PHASE6/'P6_02_test_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# 图6-3：选定阈值下测试集分类效果
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=['Precision', 'Recall', 'F1'], y=[test_precision, test_recall, test_f1], ax=ax)
ax.bar_label(ax.containers[0], fmt='%.4f', padding=3)
ax.set(xlabel='', ylabel='指标值', title=f'图6-3 测试集最终分类效果（阈值={BEST_THRESHOLD:.2f}）')
ax.set_ylim(0, 1)

fig.tight_layout()
fig.savefig(OUT_PHASE6/'P6_03_test_classification_metrics.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

分析结果：
* 测试集正样本率为32.24%。仅使用用户历史long_view_rate的baseline ROC-AUC为0.6684。LightGBM为0.7329，较基线高0.0645。
* 使用验证集选定的0.30阈值后，测试集Precision为46.88%、Recall为74.23%、F1为0.5747，Lift为1.4541。